# KA SS 26 · Challenge 2 — **Stage 10: seed-erosion probe**

Kaggle notebook. Run **interactively** on `GPU T4 x2`, `Persistence: Files only`, `Internet: On`.

### The one question this answers

Stage 8 trained two members and the ensemble was not where the gain came from:

| member | `SEED_EROSION_FRAC` | seed | crop | local F1 |
|---|---|---|---|---|
| n1 | 0.45 | 2026 | 448×576 | 0.7962 |
| n2 | **0.58** | 43 | 400×512 | **0.8055** |
| ens(n1,n2) | — | — | — | 0.8073 |

**+0.0093 from one member to the other** — larger than the SimSiam adaptation, larger than any
ensemble built so far (Stage 7's three members bought +0.0036; Stage 9's five-member cross-family
blend bought −0.0001). And it points where every diagnostic has pointed: predictions/image ran
107.8 → 111.3 against a GT mean of **125.2**. Still under-counting by ~11%.

Raising the erosion fraction shrinks each training seed core, which separates touching cells and
produces more instances. `t_seed` is already a confirmed *interior* optimum (0.65 won in Stage 5
with a grid topping at 0.65, and won again in Stage 7 with 0.90 above it), so the decode threshold
is not the binding constraint. The **target** is. Nobody has tested past 0.58.

### Why this run is designed the way it is

n1 → n2 changed **three** things at once: erosion, seed, and crop. That gain is confounded. Every
probe here fixes **seed 43 and crop 400×512** — n2's values — so the only variable is erosion and
the result is an actual curve.

| probe | erosion | why |
|---|---|---|
| `e70` | 0.70 | extends the trend furthest; run first because it is the most informative single point |
| `e65` | 0.65 | fills in between 0.58 and 0.70 |
| `e80` | 0.80 | brackets the far side — if this collapses, the optimum is bracketed |
| `e45` | 0.45 | control at n2's seed/crop, de-confounding the original n1→n2 comparison |

Budget-guarded and ordered by information value: if time runs out after two, the two that ran are
the two worth having.

### What it does not do

No new ensemble members for their own sake, no detector, no architecture change. Stage 9 measured
the ensemble return at −0.0001 and four independent Mask R-CNN runs capped ~0.03 below watershed.
The ensemble here is opportunistic only: once the maps are cached, greedy forward selection over
**all** available members (Stage 8's n1/n2 plus everything trained here) is nearly free.

### Reading the result

Public differences under ~0.008 are inside the noise of a 91-image split — Stage 5, 7, 8 and 9
landed at 0.5293 / 0.5259 / 0.5294 / 0.5286 across four genuinely different models. Trust the
**local curve shape** here, not a single public score. If F1 rises to 0.70 and falls at 0.80, that
is a real optimum; if it is flat, the knob is exhausted and no further probe will help.

## Before Run All

1. **Session options → Accelerator: GPU T4 x2, Internet: On, Persistence: Files only.**
2. **Add Input → Competitions → KA SS 26: Challenge 2.**
3. **Add Input → Datasets → `stage_8_nonssl`** (supplies n1/n2 maps for the ensemble step and
   `val_ids.json` for the split check).
4. Run interactively — **not** Save & Run All, which is what killed Stage 3.
5. `RESET_WORKDIR` stays **False**.

Cells 3 and 8 are the fail-fast gates and clear inside two minutes. Download the checkpoints from
`/kaggle/working/ckpt` as they appear.

In [1]:
# ── 1 · Imports and environment ───────────────────────────────────────────────
import os, sys, gc, json, math, time, random, shutil, subprocess, warnings, itertools
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import scipy.ndimage as ndi
from scipy.ndimage import distance_transform_edt, binary_dilation
from skimage.segmentation import watershed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

try:
    import cv2
    HAVE_CV2 = True
except ImportError:
    HAVE_CV2 = False

try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = False

def amp_autocast():
    return _autocast("cuda", dtype=torch.float16) if _AMP_NEW else _autocast()

def make_scaler():
    return _GradScaler("cuda") if _AMP_NEW else _GradScaler()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2."

T_START = time.time()
def elapsed():
    return time.time() - T_START

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h {(sec % 3600) // 60:02d}m {sec % 60:02d}s"

print("torch", torch.__version__, "| timm", timm.__version__, "| opencv", HAVE_CV2)
print("gpu:", torch.cuda.get_device_properties(0).name)

torch 2.10.0+cu128 | timm 1.0.26 | opencv True
gpu: Tesla T4


In [2]:
# ── 2 · Configuration ─────────────────────────────────────────────────────────
class CFG:
    RUN_TAG = "stage10_erosion"

    SMOKE_TEST = False      # ~10 min rehearsal: 1 probe, 3 epochs, tiny grids
    RESUME = True
    RESET_WORKDIR = False   # DANGER: wipes ckpt/ and maps/. Never True while recovering.
    STRICT_SPLIT = True

    # ── wall-clock budgets ────────────────────────────────────────────────────
    TOTAL_BUDGET_SEC  = 5.0 * 3600
    MEMBER_BUDGET_SEC = 45 * 60
    FINAL_RESERVE_SEC = 45 * 60

    IMG_H, IMG_W = 520, 704
    PAD_MULT = 32

    # ── split: must reproduce Stage 7/8 exactly or the ensemble is dishonest ──
    SPLIT_SEED = 2026
    GROUP_SPLIT = True
    DUP_CORR_THR = 0.72
    N_FOLDS = 5
    FOLD = 0
    EXPECT_TRAIN, EXPECT_VAL, EXPECT_GROUPS = 338, 86, 421

    ENCODER = "convnext_tiny.fb_in22k_ft_in1k"
    ENCODER_FALLBACKS = ("convnext_tiny", "resnet50", "resnet34")
    DECODER_CHANNELS = (256, 128, 64, 32)
    OUT_CHANNELS = 2

    BORDER_DILATION = 2
    BORDER_WEIGHT = 4.0

    # ── supervised schedule: Stage 5, unchanged ───────────────────────────────
    SUP_EPOCHS = 70
    SUP_BATCH = 8
    SUP_LR = 3e-4
    SUP_ENCODER_LR_MULT = 0.25
    SUP_WD = 1e-4
    FREEZE_ENCODER_EPOCHS = 2
    WARMUP_STEPS = 150
    STEPS_PER_EPOCH = 42
    EVAL_START = 24
    EVAL_EVERY = 2
    GRAD_CLIP = 5.0

    # SEED AND CROP ARE FIXED AT n2's VALUES ACROSS EVERY PROBE.
    # n1 -> n2 moved erosion, seed and crop together, so its +0.0093 is confounded.
    # Holding the other two still is what turns this into a curve instead of anecdotes.
    PROBE_SEED = 43
    PROBE_CROP = (400, 512)
    MEMBERS = [
        dict(name="e70", erosion=0.70),   # most informative single point - runs first
        dict(name="e65", erosion=0.65),
        dict(name="e80", erosion=0.80),
        dict(name="e45", erosion=0.45),   # control: de-confounds the original n1 vs n2
    ]

    AUG_HFLIP = 0.5
    AUG_VFLIP = 0.5
    AUG_SCALE = (0.85, 1.20)
    AUG_BRIGHT = 0.15
    AUG_CONTRAST = 0.15
    AUG_NOISE = 0.02

    TTA_FLIPS = True
    TTA_SCALES = (0.85, 1.0, 1.15)

    # score_thr was degenerate in Stage 5 and chose 0.0 unaided in Stages 7-9. Axis dropped.
    # min_area reaches down to 30 because higher erosion produces smaller seeds, and
    # GT area p1 is 53.
    T_CELL_GRID = [0.40, 0.45, 0.50, 0.55, 0.60]
    T_SEED_GRID = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
    MIN_AREA_GRID = [30, 50, 70, 90, 110]
    SCORE_THR_GRID = [0.0]
    COARSE_TC = [0.45, 0.50, 0.55]
    COARSE_TS = [0.55, 0.60, 0.65, 0.70]
    COARSE_MA = [50, 90]
    SWEEP_SUBSET = 86
    ROBUST_TOL = 0.004

    DEFAULT_POST = dict(t_cell=0.50, t_seed=0.65, min_area=70, score_thr=0.0)
    SUB_INDEX_COL = "idx"

    STAGE8_TAG = "stage8_nonssl"    # n1 / n2 maps come from the attached dataset


if CFG.SMOKE_TEST:
    CFG.SUP_EPOCHS = 3
    CFG.EVAL_START = 1
    CFG.EVAL_EVERY = 1
    CFG.MEMBERS = CFG.MEMBERS[:1]
    CFG.TTA_SCALES = (1.0,)
    CFG.T_CELL_GRID = [0.45, 0.50]
    CFG.T_SEED_GRID = [0.60, 0.65]
    CFG.MIN_AREA_GRID = [50, 90]
    CFG.COARSE_TC = [0.50]
    CFG.COARSE_TS = [0.60, 0.65]
    CFG.SWEEP_SUBSET = 12
    CFG.MEMBER_BUDGET_SEC = 10 * 60
    CFG.FINAL_RESERVE_SEC = 12 * 60
    print(">>> SMOKE TEST - results are meaningless, this only proves the pipeline runs")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SPLIT_SEED)
STATS = {"run_tag": CFG.RUN_TAG, "smoke_test": CFG.SMOKE_TEST}
print("run tag:", CFG.RUN_TAG)
print("probes:", [(m["name"], m["erosion"]) for m in CFG.MEMBERS],
      f"| all at seed {CFG.PROBE_SEED}, crop {CFG.PROBE_CROP}")

run tag: stage10_erosion
probes: [('e70', 0.7), ('e65', 0.65), ('e80', 0.8), ('e45', 0.45)] | all at seed 43, crop (400, 512)


In [3]:
# ── 3 · Paths, inputs, encoder preflight ──────────────────────────────────────
WORK = Path("/kaggle/working")
CKPT, ART, CACHE, MAPS = WORK / "ckpt", WORK / "artifacts", WORK / "cache", WORK / "maps"

if CFG.RESET_WORKDIR:
    print("!! RESET_WORKDIR is True - clearing ckpt/, artifacts/, maps/")
    for p in (CKPT, ART, MAPS):
        if p.exists():
            shutil.rmtree(p)
for p in (CKPT, ART, CACHE, MAPS):
    p.mkdir(parents=True, exist_ok=True)


def find_dataset_root():
    base = Path("/kaggle/input")
    assert base.exists(), "No /kaggle/input. Add Input -> Competitions -> KA SS 26: Challenge 2."
    for csv_path in sorted(base.rglob("train.csv")):
        d = csv_path.parent
        if (d / "train").is_dir() and (d / "test").is_dir():
            return d
    raise FileNotFoundError("Competition data not found. Add Input -> Competitions.")

BASE = find_dataset_root()
TRAIN_DIR, TEST_DIR, TRAIN_CSV = BASE / "train", BASE / "test", BASE / "train.csv"
print("data root:", BASE)
print("train", len(list(TRAIN_DIR.glob('*.png'))), "| test", len(list(TEST_DIR.glob('*.png'))))

# Stage 8 maps are optional - without them this still runs, just without n1/n2 in the pool.
S8_DIR = None
for cand in sorted(Path("/kaggle/input").glob("*")):
    hit = next(iter(cand.rglob(f"{CFG.STAGE8_TAG}_*_val.npz")), None)
    if hit is not None:
        S8_DIR = hit.parent
        break
if S8_DIR is None:
    print("!! stage_8_nonssl not attached - n1/n2 will not be in the ensemble pool.")
    STAGE8_MEMBERS = []
else:
    STAGE8_MEMBERS = sorted({p.name.split("_")[2]
                             for p in S8_DIR.glob(f"{CFG.STAGE8_TAG}_*_val.npz")})
    print("stage 8 dir:", S8_DIR, "| members:", STAGE8_MEMBERS)


def resolve_encoder():
    for name in [CFG.ENCODER] + list(CFG.ENCODER_FALLBACKS):
        try:
            m = timm.create_model(name, pretrained=True, features_only=True, in_chans=3)
            ch = m.feature_info.channels()
            del m; gc.collect()
            print(f"encoder ok: {name} | feature channels {ch}")
            return name
        except Exception as exc:
            print(f"  [{name}] unavailable: {type(exc).__name__}: {str(exc)[:160]}")
    raise RuntimeError("No pretrained encoder. Turn Internet ON in Session options.")

CFG.ENCODER = resolve_encoder()
STATS["encoder"] = CFG.ENCODER

data root: /kaggle/input/competitions/ka-ss-26-challenge-2/cells_segmentation
train 424 | test 182
stage 8 dir: /kaggle/input/datasets/ghostylicious/stage-8-nonssl | members: ['n1', 'n2']


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

encoder ok: convnext_tiny.fb_in22k_ft_in1k | feature channels [96, 192, 384, 768]


In [4]:
# ── 4 · Labels and RLE codec (byte-identical to Stages 5/7/8) ─────────────────
df_raw = pd.read_csv(TRAIN_CSV)
ID_COL = next(c for c in df_raw.columns if c.lower() in ("id", "image_id", "imageid"))
ANN_COL = next(c for c in df_raw.columns
               if c.lower() in ("annotation", "encodedpixels", "rle", "segmentation"))

TRAIN_IDS = sorted(p.stem for p in TRAIN_DIR.glob("*.png"))
TEST_IDS = sorted(p.stem for p in TEST_DIR.glob("*.png"))
with Image.open(TRAIN_DIR / f"{TRAIN_IDS[0]}.png") as im:
    CFG.IMG_W, CFG.IMG_H = im.size
PAD_H = int(math.ceil(CFG.IMG_H / CFG.PAD_MULT) * CFG.PAD_MULT)
PAD_W = int(math.ceil(CFG.IMG_W / CFG.PAD_MULT) * CFG.PAD_MULT)
NPIX = CFG.IMG_H * CFG.IMG_W


def rle_decode_flat(rle, h, w, order="C"):
    if not isinstance(rle, str) or not rle.strip() or rle.strip() == "-1":
        return np.empty(0, dtype=np.int64)
    values = np.asarray(rle.split(), dtype=np.int64)
    starts, lengths = values[0::2] - 1, values[1::2]
    idx = np.concatenate([np.arange(s, s + l) for s, l in zip(starts, lengths)])
    return idx if order == "C" else (idx % h) * w + (idx // h)


def rle_encode(mask_flat_u8):
    pixels = np.concatenate([[0], mask_flat_u8, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(int(x)) for x in runs)


def flat_to_rle(flat):
    mask = np.zeros(NPIX, dtype=np.uint8)
    mask[flat] = 1
    return rle_encode(mask)


probe_df = df_raw[df_raw[ANN_COL].astype(str) != "-1"].sample(min(200, len(df_raw)), random_state=0)
order_report = {}
for order in ("C", "F"):
    single = []
    for rle in probe_df[ANN_COL].astype(str):
        flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, order)
        if flat.size == 0 or flat.max() >= NPIX:
            continue
        m = np.zeros(NPIX, bool); m[flat] = True
        single.append(ndi.label(m.reshape(CFG.IMG_H, CFG.IMG_W))[1] == 1)
    order_report[order] = float(np.mean(single) * 100) if single else 0.0
RLE_ORDER = max(order_report, key=lambda o: order_report[o])
print("RLE_ORDER =", RLE_ORDER, {k: round(v, 1) for k, v in order_report.items()})
assert order_report[RLE_ORDER] > 90

INST = defaultdict(list)
for image_id, rle in zip(df_raw[ID_COL].astype(str), df_raw[ANN_COL].astype(str)):
    flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
    if flat.size:
        INST[image_id].append(np.unique(flat))
counts = np.array([len(INST[i]) for i in TRAIN_IDS])
areas = np.array([len(f) for i in TRAIN_IDS for f in INST[i]])
STATS["gt_mean_instances"] = float(counts.mean())
print(f"instances {int(counts.sum())} | per image mean {counts.mean():.1f} "
      f"p50 {int(np.percentile(counts, 50))} max {int(counts.max())}")
print(f"instance area p1 {int(np.percentile(areas, 1))} p50 {int(np.percentile(areas, 50))}")

bad = 0
for image_id in TRAIN_IDS[:40]:
    for flat in INST[image_id]:
        if not np.array_equal(
                np.unique(rle_decode_flat(flat_to_rle(flat), CFG.IMG_H, CFG.IMG_W, RLE_ORDER)), flat):
            bad += 1
assert bad == 0, f"RLE round-trip lossy on {bad} instances."
print(">>> RLE round-trip exact on 40 images.")

RLE_ORDER = C {'C': 91.5, 'F': 0.0}
instances 53065 | per image mean 125.2 p50 47 max 782
instance area p1 53 p50 215
>>> RLE round-trip exact on 40 images.


In [5]:
# ── 5 · Images and per-erosion target sets ────────────────────────────────────
def load_gray_u8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.uint8)

IMAGES = {i: load_gray_u8(TRAIN_DIR / f"{i}.png") for i in TRAIN_IDS}
MEAN = float(np.mean([IMAGES[i].mean() for i in TRAIN_IDS]) / 255.0)
STD = float(np.mean([IMAGES[i].std() for i in TRAIN_IDS]) / 255.0)
print(f"loaded {len(IMAGES)} images | mean/std {MEAN:.4f}/{STD:.4f}")


def build_targets_for(image_id, frac):
    h, w = CFG.IMG_H, CFG.IMG_W
    fg = np.zeros((h, w), np.uint8)
    seed = np.zeros((h, w), np.uint8)
    overlap = np.zeros((h, w), np.uint16)
    for flat in INST[image_id]:
        ys, xs = np.divmod(flat, w)
        y0, y1 = ys.min(), ys.max() + 1
        x0, x1 = xs.min(), xs.max() + 1
        sub = np.zeros((y1 - y0, x1 - x0), bool)
        sub[ys - y0, xs - x0] = True
        fg[y0:y1, x0:x1] |= sub
        dist = distance_transform_edt(np.pad(sub, 1))[1:-1, 1:-1]
        dmax = dist.max()
        if dmax > 0:
            core = dist > frac * dmax
            if not core.any():          # a high frac can erase a small cell's core entirely -
                core = dist >= dmax     # this keeps at least the peak pixel, so no cell is lost
            seed[y0:y1, x0:x1] |= core
        pad = CFG.BORDER_DILATION
        ys0, ys1 = max(0, y0 - pad), min(h, y1 + pad)
        xs0, xs1 = max(0, x0 - pad), min(w, x1 + pad)
        big = np.zeros((ys1 - ys0, xs1 - xs0), bool)
        big[y0 - ys0:y1 - ys0, x0 - xs0:x1 - xs0] = sub
        overlap[ys0:ys1, xs0:xs1] += binary_dilation(big, iterations=pad).astype(np.uint16)
    return np.stack([fg, seed, (overlap >= 2).astype(np.uint8)], 0)


TARGETS = None
_TARGET_KEY = None
SEED_PCT = {}


def load_targets(frac):
    global TARGETS, _TARGET_KEY
    if _TARGET_KEY == frac:
        return
    TARGETS = None
    gc.collect()
    path = CACHE / f"targets_e{int(round(frac * 100)):03d}.npz"
    if CFG.RESUME and path.exists():
        with np.load(path) as z:
            TARGETS = {k: z[k] for k in z.files}
        print(f"targets (erosion {frac}) from cache")
    else:
        t0 = time.time()
        TARGETS = {i: build_targets_for(i, frac) for i in TRAIN_IDS}
        np.savez_compressed(path, **TARGETS)
        print(f"targets (erosion {frac}) built in {time.time() - t0:.0f}s")
    _TARGET_KEY = frac
    pct = float(np.mean([TARGETS[i][1].mean() for i in TRAIN_IDS]) * 100)
    SEED_PCT[frac] = pct
    # This is the knob doing its work: seed coverage must shrink as erosion rises.
    print(f"  seed channel covers {pct:.2f}% of pixels at erosion {frac}")


load_targets(CFG.MEMBERS[0]["erosion"])

loaded 424 images | mean/std 0.5018/0.0522
targets (erosion 0.7) built in 24s
  seed channel covers 1.56% of pixels at erosion 0.7


In [6]:
# ── 6 · Grouped split + identity check against Stage 8 ────────────────────────
TH_H, TH_W = 65, 88


def thumbnail_vec(arr):
    a = np.asarray(Image.fromarray(arr).resize((TH_W, TH_H), Image.BILINEAR), np.float32).ravel()
    a -= a.mean()
    n = np.linalg.norm(a)
    return a / n if n > 0 else a

THUMBS = np.stack([thumbnail_vec(IMAGES[i]) for i in TRAIN_IDS])
CORR = THUMBS @ THUMBS.T
np.fill_diagonal(CORR, 0.0)

adj = CORR >= CFG.DUP_CORR_THR
group_of = -np.ones(len(TRAIN_IDS), int)
g = 0
for i in range(len(TRAIN_IDS)):
    if group_of[i] >= 0:
        continue
    stack, group_of[i] = [i], g
    while stack:
        u = stack.pop()
        for v in np.nonzero(adj[u])[0]:
            if group_of[v] < 0:
                group_of[v] = g
                stack.append(v)
    g += 1
n_groups = g

rng = np.random.RandomState(CFG.SPLIT_SEED)
order = rng.permutation(n_groups)
fold_of_group = {int(gid): int(k % CFG.N_FOLDS) for k, gid in enumerate(order)}
val_mask = np.array([fold_of_group[int(x)] == CFG.FOLD for x in group_of])
train_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if not m]
val_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if m]

vi, ti = np.nonzero(val_mask)[0], np.nonzero(~val_mask)[0]
max_corr = float(CORR[np.ix_(vi, ti)].max())
print(f"groups {n_groups} | train/val {len(train_ids)}/{len(val_ids)} | max corr {max_corr:.3f}")
assert max_corr < CFG.DUP_CORR_THR, "Near-duplicate leaked across the split."

got = (len(train_ids), len(val_ids), n_groups)
want = (CFG.EXPECT_TRAIN, CFG.EXPECT_VAL, CFG.EXPECT_GROUPS)
if got != want and CFG.STRICT_SPLIT:
    raise RuntimeError(f"SPLIT MISMATCH: got {got}, Stages 7/8 had {want}. "
                       "Local F1 would not be comparable and n1/n2 could not be ensembled.")

# Identity, not a count: Stage 7 matched Stage 5's configured split and still differed.
if S8_DIR is not None:
    vid = next(iter(S8_DIR.rglob("val_ids.json")), None)
    if vid is not None:
        s8_val = list(json.loads(vid.read_text())["val_ids"])
        if s8_val != list(val_ids):
            only8, only10 = set(s8_val) - set(val_ids), set(val_ids) - set(s8_val)
            raise RuntimeError(
                f"Validation sets differ from Stage 8 ({len(only8)} / {len(only10)} images).\n"
                "n1/n2 cannot join the ensemble pool - set STAGE8_MEMBERS = [] to continue alone.")
        print(">>> validation set identical to Stage 8 - n1/n2 can join the pool.")

STATS.update(n_groups=int(n_groups), max_val_train_corr=max_corr,
             n_train=len(train_ids), n_val=len(val_ids))
(ART / "val_ids.json").write_text(json.dumps(
    {"val_ids": val_ids, "n_train": len(train_ids), "n_groups": int(n_groups)}, indent=1))

groups 421 | train/val 338/86 | max corr 0.665
>>> validation set identical to Stage 8 - n1/n2 can join the pool.


1602

In [7]:
# ── 7 · Dataset and augmentation (unchanged from Stage 5/7/8) ─────────────────
def resize_pair(img, tgt, scale):
    h, w = img.shape
    nh, nw = max(64, int(round(h * scale))), max(64, int(round(w * scale)))
    if HAVE_CV2:
        img_r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        tgt_r = np.stack([cv2.resize(tgt[c], (nw, nh), interpolation=cv2.INTER_NEAREST)
                          for c in range(tgt.shape[0])])
    else:
        img_r = np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize(
            (nw, nh), Image.BILINEAR), np.float32) / 255.0
        tgt_r = np.stack([np.asarray(Image.fromarray(tgt[c]).resize((nw, nh), Image.NEAREST))
                          for c in range(tgt.shape[0])])
    return img_r.astype(np.float32), tgt_r.astype(np.uint8)


def pad_to(arr, height, width, mode="reflect"):
    ph, pw = max(0, height - arr.shape[-2]), max(0, width - arr.shape[-1])
    if ph == 0 and pw == 0:
        return arr
    pad_width = [(0, 0)] * (arr.ndim - 2) + [(0, ph), (0, pw)]
    kw = {} if mode == "reflect" else {"constant_values": 0}
    return np.pad(arr, pad_width, mode=mode, **kw)


class CellDataset(Dataset):
    def __init__(self, ids, crop, training=True):
        self.ids = list(ids)
        self.crop_h, self.crop_w = crop
        self.training = bool(training)

    def __len__(self):
        return len(self.ids)

    def augment(self, img, tgt):
        if random.random() < CFG.AUG_HFLIP:
            img, tgt = img[:, ::-1], tgt[:, :, ::-1]
        if random.random() < CFG.AUG_VFLIP:
            img, tgt = img[::-1], tgt[:, ::-1]
        img = np.ascontiguousarray(img)
        tgt = np.ascontiguousarray(tgt)
        s = random.uniform(*CFG.AUG_SCALE)
        if abs(s - 1.0) > 1e-3:
            img, tgt = resize_pair(img, tgt, s)
        img = img + random.uniform(-CFG.AUG_BRIGHT, CFG.AUG_BRIGHT) * STD
        f = 1.0 + random.uniform(-CFG.AUG_CONTRAST, CFG.AUG_CONTRAST)
        img = (img - MEAN) * f + MEAN
        img = img + np.random.randn(*img.shape).astype(np.float32) * CFG.AUG_NOISE * STD
        return np.clip(img, 0.0, 1.0).astype(np.float32), tgt

    def random_crop(self, img, tgt):
        img = pad_to(img, self.crop_h, self.crop_w)
        tgt = pad_to(tgt, self.crop_h, self.crop_w, mode="constant")
        h, w = img.shape
        y = random.randint(0, h - self.crop_h)
        x = random.randint(0, w - self.crop_w)
        return img[y:y + self.crop_h, x:x + self.crop_w], tgt[:, y:y + self.crop_h, x:x + self.crop_w]

    def __getitem__(self, index):
        image_id = self.ids[index]
        img = IMAGES[image_id].astype(np.float32) / 255.0
        tgt = TARGETS[image_id].copy()
        if self.training:
            img, tgt = self.augment(img, tgt)
            img, tgt = self.random_crop(img, tgt)
        else:
            img = pad_to(img, PAD_H, PAD_W)
            tgt = pad_to(tgt, PAD_H, PAD_W, mode="constant")
        weight = 1.0 + CFG.BORDER_WEIGHT * tgt[2].astype(np.float32)
        x = torch.from_numpy(np.ascontiguousarray(img)).unsqueeze(0).repeat(3, 1, 1)
        y = torch.from_numpy(np.ascontiguousarray(tgt[:2]).astype(np.float32))
        w = torch.from_numpy(np.ascontiguousarray(weight)).unsqueeze(0)
        return x, y, w, image_id


def normalize_batch(x):
    return (x - MEAN) / max(STD, 1e-6)


_ds = CellDataset(train_ids, CFG.PROBE_CROP, True)
_x, _y, _w, _ = _ds[0]
print("sample shapes:", tuple(_x.shape), tuple(_y.shape))
del _ds, _x, _y, _w

sample shapes: (3, 400, 512) (2, 400, 512)


In [8]:
# ── 8 · ConvNeXt-UNet ─────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


def up_to(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)


class UNet(nn.Module):
    def __init__(self, encoder_name, out_ch=2, dec=(256, 128, 64, 32), pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True, in_chans=3)
        chs = self.encoder.feature_info.channels()[-4:]
        self.stem = nn.Sequential(
            nn.Conv2d(3, dec[3], 3, padding=1, bias=False),
            nn.BatchNorm2d(dec[3]), nn.ReLU(inplace=True))
        self.d3 = ConvBlock(chs[3] + chs[2], dec[0])
        self.d2 = ConvBlock(dec[0] + chs[1], dec[1])
        self.d1 = ConvBlock(dec[1] + chs[0], dec[2])
        self.d0 = ConvBlock(dec[2], dec[3])
        self.dfull = ConvBlock(dec[3] * 2, dec[3])
        self.head = nn.Conv2d(dec[3], out_ch, 1)

    def forward(self, x):
        stem = self.stem(x)
        f1, f2, f3, f4 = self.encoder(x)[-4:]
        d = self.d3(torch.cat([up_to(f4, f3), f3], 1))
        d = self.d2(torch.cat([up_to(d, f2), f2], 1))
        d = self.d1(torch.cat([up_to(d, f1), f1], 1))
        d = self.d0(F.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False))
        d = self.dfull(torch.cat([up_to(d, stem), stem], 1))
        return self.head(d)


def loss_fn(logits, target, weight):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce = (bce * weight).mean()
    prob = torch.sigmoid(logits)
    num = 2.0 * (prob * target).sum(dim=(2, 3)) + 1.0
    den = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1.0
    return bce + (1.0 - (num / den).mean())


_m = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=False).to(DEVICE).eval()
with torch.no_grad(), amp_autocast():
    _o = _m(torch.zeros(1, 3, PAD_H, PAD_W, device=DEVICE))
print("model forward ok:", tuple(_o.shape))
assert _o.shape[-2:] == (PAD_H, PAD_W)
del _m, _o; gc.collect(); torch.cuda.empty_cache()

model forward ok: (1, 2, 544, 704)


In [9]:
# ── 9 · Decode, official metric, fast evaluation ──────────────────────────────
def watershed_labels(fg, seed, t_cell, t_seed):
    mask = fg >= t_cell
    if not mask.any():
        return np.zeros(fg.shape, np.int32), 0
    markers, n = ndi.label((seed >= t_seed) & mask)
    if n == 0:
        markers, n = ndi.label(mask)
        if n == 0:
            return np.zeros(fg.shape, np.int32), 0
    lab = watershed(-fg, markers, mask=mask).astype(np.int32)
    return lab, int(lab.max())


def label_stats(lab, fg, seed, nlab):
    flat = lab.ravel()
    area = np.bincount(flat, minlength=nlab + 1).astype(np.float64)
    denom = np.maximum(area, 1.0)
    mean_fg = np.bincount(flat, weights=fg.ravel(), minlength=nlab + 1) / denom
    mean_seed = np.bincount(flat, weights=seed.ravel(), minlength=nlab + 1) / denom
    return area, mean_fg, mean_seed


def prepare_eval(fg, seed, gt_flats, t_cell, t_seed):
    lab, nlab = watershed_labels(fg, seed, t_cell, t_seed)
    area, mean_fg, mean_seed = label_stats(lab, fg, seed, nlab)
    n_gt = len(gt_flats)
    if nlab == 0 or n_gt == 0:
        return dict(n_gt=n_gt, area=area, score=mean_seed, gi=np.empty(0, int),
                    pi=np.empty(0, int), nlab=nlab)
    flatlab = lab.ravel()
    gt_area = np.array([f.size for f in gt_flats], np.float64)
    C = np.zeros((n_gt, nlab + 1), np.int64)
    for i, idx in enumerate(gt_flats):
        C[i] = np.bincount(flatlab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    return dict(n_gt=n_gt, area=area, score=mean_seed, gi=gi[o], pi=pi[o], nlab=nlab)


def f1_filtered(prep, min_area, score_thr):
    keep = (prep["area"] >= min_area) & (prep["score"] >= score_thr)
    if keep.size:
        keep[0] = False
    n_pred = int(keep.sum())
    n_gt = prep["n_gt"]
    if n_gt == 0:
        return 1.0 if n_pred == 0 else 0.0
    if n_pred == 0:
        return 0.0
    used_g = np.zeros(n_gt, bool)
    used_p = np.zeros(keep.size, bool)
    tp = 0
    for g, p in zip(prep["gi"], prep["pi"]):
        if not keep[p] or used_g[g] or used_p[p]:
            continue
        used_g[g] = True; used_p[p] = True; tp += 1
    return 2.0 * tp / (2.0 * tp + (n_pred - tp) + (n_gt - tp))


def instances_from_maps(fg, seed, post):
    lab, nlab = watershed_labels(fg, seed, post["t_cell"], post["t_seed"])
    if nlab == 0:
        return []
    area, _, mean_seed = label_stats(lab, fg, seed, nlab)
    keep = (area >= post["min_area"]) & (mean_seed >= post.get("score_thr", 0.0))
    keep[0] = False
    flat = lab.ravel()
    order_ = np.argsort(flat, kind="stable")
    bounds = np.searchsorted(flat[order_], np.arange(nlab + 2))
    out = []
    for j in np.nonzero(keep)[0]:
        idx = order_[bounds[j]:bounds[j + 1]]
        if idx.size:
            out.append(np.sort(idx))
    return out


print("decode + metric ready")

decode + metric ready


In [10]:
# ── 10 · TTA inference and map I/O ────────────────────────────────────────────
@torch.no_grad()
def predict_maps(model, ids, image_dir, scales=None, flips=None):
    model.eval()
    scales = CFG.TTA_SCALES if scales is None else scales
    flips = CFG.TTA_FLIPS if flips is None else flips
    out = {}
    for image_id in ids:
        arr = IMAGES[image_id] if image_dir is None else load_gray_u8(image_dir / f"{image_id}.png")
        base = arr.astype(np.float32) / 255.0
        acc = np.zeros((2, CFG.IMG_H, CFG.IMG_W), np.float32)
        n = 0
        for s in scales:
            if abs(s - 1.0) < 1e-3:
                img = base
            elif HAVE_CV2:
                img = cv2.resize(base, (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))),
                                 interpolation=cv2.INTER_LINEAR)
            else:
                img = np.asarray(Image.fromarray((base * 255).astype(np.uint8)).resize(
                    (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))), Image.BILINEAR),
                    np.float32) / 255.0
            h, w = img.shape
            ph = int(math.ceil(h / CFG.PAD_MULT) * CFG.PAD_MULT)
            pw = int(math.ceil(w / CFG.PAD_MULT) * CFG.PAD_MULT)
            x = torch.from_numpy(np.ascontiguousarray(pad_to(img, ph, pw)))
            x = normalize_batch(x.unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)).to(DEVICE)
            variants = [(False, False)] if not flips else [(False, False), (True, False),
                                                           (False, True), (True, True)]
            for hf, vf in variants:
                xin = x
                if hf:
                    xin = torch.flip(xin, dims=[3])
                if vf:
                    xin = torch.flip(xin, dims=[2])
                with amp_autocast():
                    logits = model(xin)
                prob = torch.sigmoid(logits.float())
                if vf:
                    prob = torch.flip(prob, dims=[2])
                if hf:
                    prob = torch.flip(prob, dims=[3])
                p = prob[0, :, :h, :w].cpu().numpy()
                if p.shape[1:] != (CFG.IMG_H, CFG.IMG_W):
                    if HAVE_CV2:
                        p = np.stack([cv2.resize(p[c], (CFG.IMG_W, CFG.IMG_H),
                                                 interpolation=cv2.INTER_LINEAR) for c in range(2)])
                    else:
                        p = np.stack([np.asarray(Image.fromarray((p[c] * 255).astype(np.uint8)).resize(
                            (CFG.IMG_W, CFG.IMG_H), Image.BILINEAR), np.float32) / 255.0
                            for c in range(2)])
                acc += p
                n += 1
        out[image_id] = np.clip(acc / max(n, 1) * 255.0, 0, 255).astype(np.uint8)
    return out


def maps_path(member, split):
    if member in STAGE8_MEMBERS:
        return S8_DIR / f"{CFG.STAGE8_TAG}_{member}_{split}.npz"
    return MAPS / f"{CFG.RUN_TAG}_{member}_{split}.npz"


def load_maps(member, split):
    with np.load(maps_path(member, split)) as z:
        return {k: z[k] for k in z.files}


def family_mean(members, split, ids):
    acc = None
    for m in members:
        mp = load_maps(m, split)
        if acc is None:
            acc = {i: mp[i].astype(np.float32) for i in ids}
        else:
            for i in ids:
                acc[i] += mp[i].astype(np.float32)
        del mp; gc.collect()
    for i in ids:
        acc[i] /= (len(members) * 255.0)
    return acc


def maps_to_float(u8):
    return u8[0].astype(np.float32) / 255.0, u8[1].astype(np.float32) / 255.0


print("inference helpers ready")

inference helpers ready


In [11]:
# ── 11 · Train one probe ──────────────────────────────────────────────────────
def quick_val_f1(model, ids, post):
    maps = predict_maps(model, ids, None, scales=(1.0,), flips=False)
    return float(np.mean([
        f1_filtered(prepare_eval(*maps_to_float(maps[i]), INST[i], post["t_cell"], post["t_seed"]),
                    post["min_area"], post["score_thr"]) for i in ids]))


def train_probe(spec):
    tag = f"{CFG.RUN_TAG}_{spec['name']}"
    best_path = CKPT / f"{tag}_best.pt"
    if CFG.RESUME and best_path.exists():
        blob = torch.load(best_path, map_location="cpu")
        if blob.get("run_tag") == CFG.RUN_TAG and blob.get("member") == spec["name"]:
            print(f"[{spec['name']}] finished checkpoint (epoch {blob['epoch']}, "
                  f"F1 {blob['f1']:.4f}) - skipping")
            return blob["f1"]

    load_targets(spec["erosion"])
    seed_everything(CFG.PROBE_SEED)          # fixed across probes - erosion is the only variable
    model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=True).to(DEVICE)

    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW(
        [{"params": dec_params, "lr": CFG.SUP_LR},
         {"params": enc_params, "lr": CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT}],
        weight_decay=CFG.SUP_WD)

    loader = DataLoader(CellDataset(train_ids, CFG.PROBE_CROP, True), batch_size=CFG.SUP_BATCH,
                        shuffle=True, num_workers=2, pin_memory=True, drop_last=True,
                        persistent_workers=True)
    total_steps = CFG.SUP_EPOCHS * CFG.STEPS_PER_EPOCH
    scaler = make_scaler()
    best_f1, best_epoch, log = -1.0, -1, []
    t0 = time.time()

    def lr_at(step):
        if step < CFG.WARMUP_STEPS:
            return (step + 1) / max(1, CFG.WARMUP_STEPS)
        p = (step - CFG.WARMUP_STEPS) / max(1, total_steps - CFG.WARMUP_STEPS)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

    step, it = 0, iter(loader)
    for ep in range(CFG.SUP_EPOCHS):
        if time.time() - t0 > CFG.MEMBER_BUDGET_SEC:
            print(f"[{spec['name']}] budget reached at epoch {ep} - stopping")
            break
        freeze = ep < CFG.FREEZE_ENCODER_EPOCHS
        for p in enc_params:
            p.requires_grad_(not freeze)
        model.train()
        run = 0.0
        for _ in range(CFG.STEPS_PER_EPOCH):
            try:
                x, y, w, _ = next(it)
            except StopIteration:
                it = iter(loader)
                x, y, w, _ = next(it)
            x = normalize_batch(x).to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            w = w.to(DEVICE, non_blocking=True)
            mult = lr_at(step)
            opt.param_groups[0]["lr"] = CFG.SUP_LR * mult
            opt.param_groups[1]["lr"] = (CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT * mult
                                         * (0.0 if freeze else 1.0))
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = loss_fn(model(x), y, w)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update()
            run += loss.item(); step += 1
        msg = f"[{spec['name']}] ep {ep + 1:02d}/{CFG.SUP_EPOCHS} loss {run / CFG.STEPS_PER_EPOCH:.4f}"
        if ep + 1 >= CFG.EVAL_START and (ep + 1 - CFG.EVAL_START) % CFG.EVAL_EVERY == 0:
            f1 = quick_val_f1(model, val_ids, CFG.DEFAULT_POST)
            msg += f" | val F1 {f1:.4f}"
            if f1 > best_f1:
                best_f1, best_epoch = f1, ep + 1
                torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": ep + 1,
                            "f1": f1, "state_dict": model.state_dict(), "spec": spec,
                            "encoder": CFG.ENCODER}, best_path)
                msg += "  *"
            log.append(dict(member=spec["name"], erosion=spec["erosion"], epoch=ep + 1,
                            loss=run / CFG.STEPS_PER_EPOCH, f1=f1))
        print(msg + f" | {fmt(time.time() - t0)}", flush=True)

    if best_f1 < 0:
        best_f1, best_epoch = quick_val_f1(model, val_ids, CFG.DEFAULT_POST), CFG.SUP_EPOCHS
        torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": best_epoch,
                    "f1": best_f1, "state_dict": model.state_dict(), "spec": spec,
                    "encoder": CFG.ENCODER}, best_path)
    pd.DataFrame(log).to_csv(ART / f"train_log_{spec['name']}.csv", index=False)
    print(f"[{spec['name']}] best epoch {best_epoch} | quick F1 {best_f1:.4f} | "
          f"{fmt(time.time() - t0)}")
    del model, opt, loader, scaler
    gc.collect(); torch.cuda.empty_cache()
    return best_f1

In [12]:
# ── 12 · Run the probes, cache maps ───────────────────────────────────────────
def write_submission(path, maps, ids, post):
    rows, counts, fallbacks = [], [], 0
    for image_id in ids:
        m = maps[image_id]
        fg, seed = maps_to_float(m) if m.dtype == np.uint8 else (m[0], m[1])
        inst = instances_from_maps(fg, seed, post)
        if not inst:  # an image with GT and zero masks scores exactly 0.0
            fallbacks += 1
            inst = instances_from_maps(fg, seed,
                                       dict(t_cell=0.30, t_seed=0.30, min_area=0, score_thr=0.0))[:1]
            if not inst:
                h = CFG.IMG_H // 2
                inst = [np.arange(h * CFG.IMG_W, h * CFG.IMG_W + 25)]
        counts.append(len(inst))
        rows.extend((image_id, flat_to_rle(f)) for f in inst)
    frame = pd.DataFrame(rows, columns=[ID_COL, ANN_COL])
    frame.insert(0, CFG.SUB_INDEX_COL, np.arange(len(frame), dtype=np.int64))
    frame.to_csv(path, index=False)
    assert set(frame[ID_COL].astype(str)) == set(ids), "Submission is missing test images."
    assert frame[ANN_COL].notna().all() and frame[CFG.SUB_INDEX_COL].is_unique
    return dict(path=str(path), rows=len(frame), mean=float(np.mean(counts)),
                min=int(np.min(counts)), max=int(np.max(counts)), fallbacks=fallbacks)


PROBE_RESULTS = []
for spec in CFG.MEMBERS:
    if elapsed() > CFG.TOTAL_BUDGET_SEC - CFG.FINAL_RESERVE_SEC:
        print(f"!! stopping after {len(PROBE_RESULTS)} probe(s) - reserving time for the sweep")
        break
    quick = train_probe(spec)
    tag = f"{CFG.RUN_TAG}_{spec['name']}"

    if CFG.RESUME and maps_path(spec["name"], "val").exists() \
            and maps_path(spec["name"], "test").exists():
        print(f"[{spec['name']}] cached maps found")
    else:
        blob = torch.load(CKPT / f"{tag}_best.pt", map_location="cpu")
        assert blob["run_tag"] == CFG.RUN_TAG and blob["member"] == spec["name"]
        model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS,
                     pretrained=False).to(DEVICE)
        model.load_state_dict(blob["state_dict"])
        t0 = time.time()
        np.savez_compressed(MAPS / f"{tag}_val.npz", **predict_maps(model, val_ids, None))
        np.savez_compressed(MAPS / f"{tag}_test.npz",
                            **predict_maps(model, TEST_IDS, TEST_DIR))
        print(f"[{spec['name']}] TTA maps cached in {fmt(time.time() - t0)}")
        del model, blob
        gc.collect(); torch.cuda.empty_cache()

    vm = load_maps(spec["name"], "val")
    tta = float(np.mean([
        f1_filtered(prepare_eval(*maps_to_float(vm[i]), INST[i], CFG.DEFAULT_POST["t_cell"],
                                 CFG.DEFAULT_POST["t_seed"]),
                    CFG.DEFAULT_POST["min_area"], CFG.DEFAULT_POST["score_thr"])
        for i in val_ids]))
    info = write_submission(ART / f"submission_{spec['name']}.csv",
                            load_maps(spec["name"], "test"), TEST_IDS, CFG.DEFAULT_POST)
    print(f"[{spec['name']}] erosion {spec['erosion']} | TTA F1 (default post) {tta:.4f} | "
          f"test preds/image {info['mean']:.1f}")
    PROBE_RESULTS.append(dict(member=spec["name"], erosion=spec["erosion"], quick_f1=quick,
                              tta_f1=tta, seed_pct=SEED_PCT.get(spec["erosion"]), **info))
    del vm; gc.collect()
    print("   >>> download /kaggle/working/ckpt/%s_best.pt now <<<\n" % tag, flush=True)

assert PROBE_RESULTS, "No probe finished."
POOL = [p["member"] for p in PROBE_RESULTS] + list(STAGE8_MEMBERS)
print("ensemble pool:", POOL, "| elapsed", fmt(elapsed()))

[e70] ep 01/70 loss 1.5363 | 0h 00m 34s
[e70] ep 02/70 loss 1.3978 | 0h 00m 43s
[e70] ep 03/70 loss 1.2989 | 0h 01m 12s
[e70] ep 04/70 loss 1.2350 | 0h 01m 25s
[e70] ep 05/70 loss 1.1735 | 0h 01m 38s
[e70] ep 06/70 loss 1.1233 | 0h 01m 52s
[e70] ep 07/70 loss 1.0721 | 0h 02m 05s
[e70] ep 08/70 loss 1.0276 | 0h 02m 19s
[e70] ep 09/70 loss 0.9857 | 0h 02m 33s
[e70] ep 10/70 loss 0.9583 | 0h 02m 47s
[e70] ep 11/70 loss 0.9145 | 0h 03m 01s
[e70] ep 12/70 loss 0.8790 | 0h 03m 14s
[e70] ep 13/70 loss 0.8474 | 0h 03m 28s
[e70] ep 14/70 loss 0.8118 | 0h 03m 42s
[e70] ep 15/70 loss 0.7800 | 0h 03m 55s
[e70] ep 16/70 loss 0.7511 | 0h 04m 09s
[e70] ep 17/70 loss 0.7198 | 0h 04m 23s
[e70] ep 18/70 loss 0.6876 | 0h 04m 36s
[e70] ep 19/70 loss 0.6510 | 0h 04m 50s
[e70] ep 20/70 loss 0.6260 | 0h 05m 04s
[e70] ep 21/70 loss 0.5967 | 0h 05m 18s
[e70] ep 22/70 loss 0.5774 | 0h 05m 31s
[e70] ep 23/70 loss 0.5527 | 0h 05m 45s
[e70] ep 24/70 loss 0.5332 | val F1 0.7720  * | 0h 06m 05s
[e70] ep 25/70 loss 0

In [13]:
# ── 13 · THE EROSION CURVE ────────────────────────────────────────────────────
# This is the deliverable. Everything else in the notebook is scaffolding for it.
def sweep(maps, ids, tc_grid, ts_grid, ma_grid, label=""):
    rows, per_image = [], {}
    t0 = time.time()
    for tc in tc_grid:
        for ts in ts_grid:
            preps = [prepare_eval(maps[i][0], maps[i][1], INST[i], tc, ts) for i in ids]
            for ma in ma_grid:
                for st in CFG.SCORE_THR_GRID:
                    vals = np.array([f1_filtered(p, ma, st) for p in preps])
                    rows.append(dict(t_cell=tc, t_seed=ts, min_area=ma, score_thr=st,
                                     f1=float(vals.mean()),
                                     mean_pred=float(np.mean([
                                         int(((p["area"][1:] >= ma) & (p["score"][1:] >= st)).sum())
                                         for p in preps]))))
                    per_image[(tc, ts, ma, st)] = vals
            del preps
    if label:
        print(f"    {label} swept in {time.time() - t0:.0f}s", flush=True)
    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True), per_image


def plateau_post(fine, tc_grid, ts_grid, ma_grid):
    # Never the peak: an optimum measured on 86 images is a region, not a point.
    peak = float(fine.f1.max())
    plat = fine[fine.f1 >= peak - CFG.ROBUST_TOL]
    snap = lambda v, grid: min(grid, key=lambda g: abs(g - v))
    return dict(t_cell=snap(float(np.median(plat.t_cell)), tc_grid),
                t_seed=snap(float(np.median(plat.t_seed)), ts_grid),
                min_area=int(snap(int(np.median(plat.min_area)), ma_grid)),
                score_thr=0.0)


coarse_ids = val_ids[:CFG.SWEEP_SUBSET]
KNOWN = {"n1": 0.45, "n2": 0.58}      # Stage 8, but at seed 2026/43 and different crops
curve = []
for m in POOL:
    vm = family_mean([m], "val", coarse_ids)
    tab, _ = sweep(vm, coarse_ids, CFG.COARSE_TC, CFG.COARSE_TS, CFG.COARSE_MA, label=m)
    b = tab.iloc[0]
    ero = next((p["erosion"] for p in PROBE_RESULTS if p["member"] == m), KNOWN.get(m))
    curve.append(dict(member=m, erosion=ero, coarse_f1=float(b.f1),
                      t_cell=float(b.t_cell), t_seed=float(b.t_seed),
                      min_area=int(b.min_area), mean_pred=float(b.mean_pred),
                      controlled=m in [p["member"] for p in PROBE_RESULTS]))
    print(f"  {m}: erosion {ero} | coarse F1 {b.f1:.4f} | preds/img {b.mean_pred:.1f} "
          f"| t_seed {b.t_seed:.2f}")
    del vm; gc.collect()

CURVE = pd.DataFrame(curve).sort_values("erosion").reset_index(drop=True)
CURVE.to_csv(ART / "erosion_curve.csv", index=False)
print("\n=== EROSION CURVE (coarse, %d images) ===" % len(coarse_ids))
print(CURVE.to_string(index=False))
print(f"\nGT mean instances/image: {STATS['gt_mean_instances']:.1f}")
print("Rows with controlled=True share seed 43 and crop 400x512, so their erosion")
print("comparison is clean. n1/n2 came from Stage 8 at different seeds/crops - they")
print("anchor the picture but are not part of the controlled curve.")

ctrl = CURVE[CURVE.controlled]
if len(ctrl) >= 2:
    top = ctrl.sort_values("coarse_f1", ascending=False).iloc[0]
    lo, hi = ctrl.erosion.min(), ctrl.erosion.max()
    if abs(top.erosion - hi) < 1e-9:
        print(f"\n!! best controlled probe is at the TOP of the tested range ({hi}) - "
              "the optimum may lie higher still; a further probe is warranted if time allows.")
    elif abs(top.erosion - lo) < 1e-9:
        print(f"\n!! best controlled probe is at the BOTTOM ({lo}) - "
              "higher erosion is not the answer; the knob is exhausted.")
    else:
        print(f"\n>>> interior optimum at erosion {top.erosion} - the curve is bracketed.")
    if ctrl.coarse_f1.max() - ctrl.coarse_f1.min() < 0.004:
        print("!! spread across the controlled probes is under 0.004 - that is inside noise. "
              "Treat the erosion knob as exhausted and do not spend more runs on it.")

    e70 swept in 23s
  e70: erosion 0.7 | coarse F1 0.8086 | preds/img 108.4 | t_seed 0.65
    e65 swept in 23s
  e65: erosion 0.65 | coarse F1 0.8049 | preds/img 108.3 | t_seed 0.55
    e80 swept in 22s
  e80: erosion 0.8 | coarse F1 0.8039 | preds/img 103.6 | t_seed 0.55
    e45 swept in 24s
  e45: erosion 0.45 | coarse F1 0.7933 | preds/img 111.1 | t_seed 0.70
    n1 swept in 22s
  n1: erosion 0.45 | coarse F1 0.7954 | preds/img 113.5 | t_seed 0.70
    n2 swept in 23s
  n2: erosion 0.58 | coarse F1 0.8035 | preds/img 108.4 | t_seed 0.60

=== EROSION CURVE (coarse, 86 images) ===
member  erosion  coarse_f1  t_cell  t_seed  min_area  mean_pred  controlled
   e45     0.45   0.793267    0.55    0.70        90 111.139535        True
    n1     0.45   0.795356    0.45    0.70        90 113.453488       False
    n2     0.58   0.803469    0.50    0.60        90 108.372093       False
   e65     0.65   0.804873    0.50    0.55        90 108.255814        True
   e70     0.70   0.808552    0

In [14]:
# ── 14 · Greedy forward selection, then a full sweep on the winner ────────────
# Combos are RANKED at one fixed reference setting (cheap), and only the winner gets
# the full grid. Stage 7 bought +0.0036 from two extra members and Stage 9 bought
# -0.0001 from five, so this stays opportunistic rather than becoming the plan.
REF = CURVE.sort_values("coarse_f1", ascending=False).iloc[0]
REF_POST = dict(t_cell=float(REF.t_cell), t_seed=float(REF.t_seed),
                min_area=int(REF.min_area), score_thr=0.0)
print("reference post for combo ranking:", REF_POST)


def combo_f1(members, ids, post):
    vm = family_mean(members, "val", ids)
    out = float(np.mean([
        f1_filtered(prepare_eval(vm[i][0], vm[i][1], INST[i], post["t_cell"], post["t_seed"]),
                    post["min_area"], post["score_thr"]) for i in ids]))
    del vm; gc.collect()
    return out


chosen = [str(REF.member)]
best_combo_f1 = combo_f1(chosen, val_ids, REF_POST)
print(f"start: {chosen} -> {best_combo_f1:.4f}")
improved = True
while improved and len(chosen) < len(POOL):
    improved = False
    cands = []
    for m in POOL:
        if m in chosen:
            continue
        f1 = combo_f1(chosen + [m], val_ids, REF_POST)
        cands.append((f1, m))
        print(f"  + {m}: {f1:.4f}")
    if cands:
        f1, m = max(cands)
        if f1 > best_combo_f1 + 1e-4:
            chosen.append(m)
            best_combo_f1 = f1
            improved = True
            print(f"  -> accepted {m} | {chosen} = {best_combo_f1:.4f}")
        else:
            print(f"  -> no member improves on {best_combo_f1:.4f}; stopping")

print(f"\nselected members: {chosen} | reference F1 {best_combo_f1:.4f}")
STATS["chosen_members"] = chosen

print("\nfull sweep on the selected combination ...", flush=True)
VAL_BEST = family_mean(chosen, "val", val_ids)
fine, PER = sweep(VAL_BEST, val_ids, CFG.T_CELL_GRID, CFG.T_SEED_GRID,
                  CFG.MIN_AREA_GRID, label="final")
fine.to_csv(ART / "sweep_final.csv", index=False)
POST = plateau_post(fine, CFG.T_CELL_GRID, CFG.T_SEED_GRID, CFG.MIN_AREA_GRID)
KEY = (POST["t_cell"], POST["t_seed"], POST["min_area"], POST["score_thr"])
if KEY not in PER:
    b = fine.iloc[0]
    POST = dict(t_cell=float(b.t_cell), t_seed=float(b.t_seed),
                min_area=int(b.min_area), score_thr=0.0)
    KEY = (POST["t_cell"], POST["t_seed"], POST["min_area"], POST["score_thr"])
LOCAL_F1 = float(PER[KEY].mean())
print(fine.head(10).to_string(index=False))
print(f"\nplateau centre {POST} | local F1 {LOCAL_F1:.4f} (peak {fine.f1.max():.4f})")
print("reference: Stage 8 ens 0.8073 -> 0.5294 public | Stage 9 blend 0.8072 -> 0.5286")

reference post for combo ranking: {'t_cell': 0.45, 't_seed': 0.65, 'min_area': 90, 'score_thr': 0.0}
start: ['e70'] -> 0.8086
  + e65: 0.8081
  + e80: 0.8023
  + e45: 0.8062
  + n1: 0.8074
  + n2: 0.8076
  -> no member improves on 0.8086; stopping

selected members: ['e70'] | reference F1 0.8086

full sweep on the selected combination ...
    final swept in 75s
 t_cell  t_seed  min_area  score_thr       f1  mean_pred
   0.45    0.65        90        0.0 0.808552 108.395349
   0.45    0.60        90        0.0 0.808400 108.837209
   0.40    0.60        90        0.0 0.807721 109.395349
   0.45    0.60        70        0.0 0.807672 111.232558
   0.40    0.65        90        0.0 0.807533 108.965116
   0.45    0.70        90        0.0 0.807262 107.720930
   0.50    0.65        90        0.0 0.807231 107.895349
   0.45    0.65       110        0.0 0.807217 105.662791
   0.40    0.65       110        0.0 0.807127 106.313953
   0.50    0.60        90        0.0 0.806999 108.279070

plateau 

In [15]:
# ── 15 · RLE parity, then write submission.csv ────────────────────────────────
parity = []
for image_id in val_ids[:25]:
    inst = instances_from_maps(VAL_BEST[image_id][0], VAL_BEST[image_id][1], POST)
    if not inst:
        parity.append(0.0); continue
    rt = [rle_decode_flat(flat_to_rle(f), CFG.IMG_H, CFG.IMG_W, RLE_ORDER) for f in inst]
    lab = np.zeros(NPIX, np.int32)
    for j, f in enumerate(rt, 1):
        lab[f] = j
    nlab = len(rt)
    gt = INST[image_id]
    gt_area = np.array([f.size for f in gt], np.float64)
    area = np.bincount(lab, minlength=nlab + 1).astype(np.float64)
    C = np.zeros((len(gt), nlab + 1), np.int64)
    for i, idx in enumerate(gt):
        C[i] = np.bincount(lab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    ug, up, tp = set(), set(), 0
    for g_, p_ in zip(gi[o], pi[o]):
        if g_ in ug or p_ in up:
            continue
        ug.add(g_); up.add(p_); tp += 1
    parity.append(2.0 * tp / (2.0 * tp + (nlab - tp) + (len(gt) - tp)))

direct = float(PER[KEY][:25].mean())
delta = abs(direct - float(np.mean(parity)))
print(f"parity | in-memory {direct:.4f} | after RLE {np.mean(parity):.4f} | delta {delta:.5f}")
print("!! encoding is losing score - fix before submitting" if delta > 0.002
      else ">>> encoding is lossless")
STATS["parity_delta"] = float(delta)
del VAL_BEST; gc.collect()

TEST_BEST = family_mean(chosen, "test", TEST_IDS)
info = write_submission(WORK / "submission.csv", TEST_BEST, TEST_IDS, POST)
shutil.copy2(WORK / "submission.csv", ART / "submission_stage10.csv")
print("\nwrote:", WORK / "submission.csv", "|", info)
print(f"predictions/image {info['mean']:.1f} vs GT mean {STATS['gt_mean_instances']:.1f} "
      f"({100 * info['mean'] / STATS['gt_mean_instances']:.0f}% of the training rate)")
print("  Stage 8 sat at 111.3 (89%). If this has not moved, higher erosion did not")
print("  fix the under-counting and the remaining error is not a seed-size problem.")
STATS["submission"] = info
del TEST_BEST; gc.collect()

parity | in-memory 0.8251 | after RLE 0.8251 | delta 0.00000
>>> encoding is lossless

wrote: /kaggle/working/submission.csv | {'path': '/kaggle/working/submission.csv', 'rows': 19549, 'mean': 107.41208791208791, 'min': 13, 'max': 605, 'fallbacks': 0}
predictions/image 107.4 vs GT mean 125.2 (86% of the training rate)
  Stage 8 sat at 111.3 (89%). If this has not moved, higher erosion did not
  fix the under-counting and the remaining error is not a seed-size problem.


0

In [16]:
# ── 16 · Report ───────────────────────────────────────────────────────────────
probe_tab = pd.DataFrame(PROBE_RESULTS)[
    ["member", "erosion", "seed_pct", "quick_f1", "tta_f1", "mean", "fallbacks"]]
lines = [
    "# Stage 10 report - seed-erosion probe",
    f"- run tag: {CFG.RUN_TAG} | encoder: {CFG.ENCODER} (plain ImageNet-21k)",
    f"- all probes at seed {CFG.PROBE_SEED}, crop {CFG.PROBE_CROP} - erosion is the only variable",
    f"- fold {CFG.FOLD}/{CFG.N_FOLDS} | train/val {len(train_ids)}/{len(val_ids)} | "
    f"groups {STATS['n_groups']} | max val-to-train corr {STATS['max_val_train_corr']:.3f}",
    "",
    "## Probes",
] + ["    " + l for l in probe_tab.to_string(index=False).splitlines()] + [
    "",
    "## Erosion curve (coarse sweep)",
] + ["    " + l for l in CURVE.to_string(index=False).splitlines()] + [
    "",
    f"- selected members: {chosen}",
    f"- chosen post-processing: {POST}",
    f"- local F1: {LOCAL_F1:.4f}",
    f"- RLE parity delta: {STATS['parity_delta']:.5f}",
    f"- test predictions/image: mean {info['mean']:.1f} "
    f"(min {info['min']}, max {info['max']}), fallbacks {info['fallbacks']}",
    f"- GT mean instances/image: {STATS['gt_mean_instances']:.1f}",
    f"- total elapsed: {fmt(elapsed())}",
    "",
    "## Prior points",
    "| stage | local | public |",
    "|---|---|---|",
    "| 5  single ImageNet U-Net | 0.8279 | 0.5293 |",
    "| 7  SSL x3 ensemble       | 0.8035 | 0.5259 |",
    "| 8  non-SSL n1+n2         | 0.8073 | 0.5294 |",
    "| 9  cross-family blend    | 0.8072 | 0.5286 |",
    f"| 10 erosion probe        | {LOCAL_F1:.4f} | ? |",
    "",
    "Those four public scores span 0.0035 across four genuinely different models - that is the",
    "noise floor of a 91-image split, not a ranking. Judge this run on the erosion curve's",
    "shape, not on a single public score.",
    "",
    "## Selection",
    "Two slots. Keep the best measured file in one and something UNCORRELATED with it in the",
    "other - the slots are scored separately and the better private one counts, so two shots",
    "that fail differently beat two that fail together.",
]
(ART / "REPORT.md").write_text("\n".join(lines))
with open(ART / "stats.json", "w") as f:
    json.dump(STATS, f, indent=2, default=str)
print("\n".join(lines))
print("\nArtifacts:")
for p in sorted(ART.iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size / 1e6:8.2f} MB")
print("\nDOWNLOAD /kaggle/working/ckpt/*.pt AND /kaggle/working/submission.csv NOW.")
print("Total elapsed:", fmt(elapsed()))

# Stage 10 report - seed-erosion probe
- run tag: stage10_erosion | encoder: convnext_tiny.fb_in22k_ft_in1k (plain ImageNet-21k)
- all probes at seed 43, crop (400, 512) - erosion is the only variable
- fold 0/5 | train/val 338/86 | groups 421 | max val-to-train corr 0.665

## Probes
    member  erosion  seed_pct  quick_f1   tta_f1       mean  fallbacks
       e70     0.70  1.561275  0.792441 0.806190 110.032967          0
       e65     0.65  1.968103  0.788588 0.804938 108.961538          0
       e80     0.80  0.778643  0.790789 0.791371 102.510989          0
       e45     0.45  4.169578  0.774600 0.790773 113.747253          0

## Erosion curve (coarse sweep)
    member  erosion  coarse_f1  t_cell  t_seed  min_area  mean_pred  controlled
       e45     0.45   0.793267    0.55    0.70        90 111.139535        True
        n1     0.45   0.795356    0.45    0.70        90 113.453488       False
        n2     0.58   0.803469    0.50    0.60        90 108.372093       False
       

## Stage 11

In [18]:
# ── A · DENSITY AUDIT — robust Stage 10 reload ────────────────────────────────

# Stage 10 final post-processing
POST_AUDIT = {
    "t_cell": 0.50,
    "t_seed": 0.65,
    "min_area": 90,
    "score_thr": 0.0,
}

# Reload e70 validation maps if VAL_BEST was deleted earlier
if "VAL_BEST" not in globals():
    candidates = [
        MAPS / "stage10_erosion_e70_val.npz",
        MAPS / f"{CFG.RUN_TAG}_e70_val.npz",
    ]
    val_map_path = next((p for p in candidates if p.exists()), None)

    if val_map_path is None:
        # fallback search so we do not depend on an exact filename
        found = sorted(MAPS.glob("*e70*val*.npz"))
        assert found, f"Could not find e70 validation maps in {MAPS}"
        val_map_path = found[0]

    print("loading e70 validation maps:", val_map_path)

    with np.load(val_map_path) as z:
        raw = {k: z[k] for k in z.files}

    # Stage 10 cached maps are uint8 probability maps.
    # Convert to the same float [0,1] representation used during evaluation.
    VAL_BEST = {
        image_id: raw[image_id].astype(np.float32) / 255.0
        for image_id in val_ids
    }

    del raw
    gc.collect()

print("VAL_BEST ready:", len(VAL_BEST), "validation images")

BINS = [
    (0, 50, "sparse <50"),
    (50, 200, "medium 50-199"),
    (200, 400, "dense 200-399"),
    (400, 10**9, "very dense >=400"),
]

rows = []

for image_id in val_ids:
    fg, seed = VAL_BEST[image_id][0], VAL_BEST[image_id][1]

    prep = prepare_eval(
        fg,
        seed,
        INST[image_id],
        POST_AUDIT["t_cell"],
        POST_AUDIT["t_seed"],
    )

    f1 = f1_filtered(
        prep,
        POST_AUDIT["min_area"],
        POST_AUDIT["score_thr"],
    )

    keep = (
        (prep["area"] >= POST_AUDIT["min_area"])
        & (prep["score"] >= POST_AUDIT["score_thr"])
    )

    if keep.size:
        keep[0] = False

    n_pred = int(keep.sum())
    n_gt = int(prep["n_gt"])

    # Official per-image F1:
    # F1 = 2TP / (N_pred + N_gt)
    tp = f1 * (n_pred + n_gt) / 2.0

    rows.append(
        dict(
            id=image_id,
            n_gt=n_gt,
            n_pred=n_pred,
            f1=f1,
            tp=tp,
            fn=n_gt - tp,
            fp=n_pred - tp,
        )
    )

AUDIT = pd.DataFrame(rows)

edges = [0, 50, 200, 400, 10**9]
labels = [b[2] for b in BINS]

AUDIT["bin"] = pd.cut(
    AUDIT.n_gt,
    edges,
    labels=labels,
    right=False,
)

tab = AUDIT.groupby("bin", observed=False).agg(
    images=("id", "size"),
    mean_f1=("f1", "mean"),
    mean_gt=("n_gt", "mean"),
    mean_pred=("n_pred", "mean"),
    mean_fn=("fn", "mean"),
    mean_fp=("fp", "mean"),
)

tab["pred_over_gt"] = tab.mean_pred / tab.mean_gt
tab["share_of_images"] = tab.images / len(AUDIT)

print("\n=== DENSITY AUDIT ===")
print(tab.round(4).to_string())

print(
    f"\noverall mean F1 {AUDIT.f1.mean():.4f} | "
    f"mean GT {AUDIT.n_gt.mean():.1f} | "
    f"mean pred {AUDIT.n_pred.mean():.1f}"
)

print("\n=== THEORETICAL PAYOFF IF EACH BIN REACHED F1 0.90 ===")

for lo, hi, name in BINS:
    m = (AUDIT.n_gt >= lo) & (AUDIT.n_gt < hi)

    if not m.any():
        continue

    lifted = AUDIT.f1.copy()
    lifted[m] = np.maximum(lifted[m], 0.90)

    gain = lifted.mean() - AUDIT.f1.mean()

    print(
        f"{name:18s} -> overall {lifted.mean():.4f} "
        f"(+{gain:.4f}) across {int(m.sum())} images"
    )

loading e70 validation maps: /kaggle/working/maps/stage10_erosion_e70_val.npz
VAL_BEST ready: 86 validation images

=== DENSITY AUDIT ===
                  images  mean_f1   mean_gt  mean_pred   mean_fn  mean_fp  pred_over_gt  share_of_images
bin                                                                                                     
sparse <50            41   0.8573   27.3902    25.9756    4.4146   3.0000        0.9484           0.4767
medium 50-199         28   0.7569   91.8929    90.6071   22.9643  21.6786        0.9860           0.3256
dense 200-399         11   0.7792  320.6364   289.7273   81.0909  50.1818        0.9036           0.1279
very dense >=400       6   0.7518  476.8333   415.0000  135.6667  73.8333        0.8703           0.0698

overall mean F1 0.8072 | mean GT 117.3 | mean pred 107.9

=== THEORETICAL PAYOFF IF EACH BIN REACHED F1 0.90 ===
sparse <50         -> overall 0.8426 (+0.0353) across 41 images
medium 50-199      -> overall 0.8556 (+0.0484) across 

In [19]:
# ── B · SEED SURVIVAL UNDER ZOOM-OUT ─────────────────────────────────────────

print("building erosion-0.70 targets ...")
load_targets(0.70)

SCALES = (0.85, 0.80, 0.75, 0.70, 0.68)

audit_ids = train_ids[:40]

print(f"testing {len(audit_ids)} training images\n")

for s in SCALES:
    base_cc = 0
    nn_cc = 0
    area_cc = 0

    base_px = 0
    nn_px = 0
    area_px = 0

    for image_id in audit_ids:
        seed = TARGETS[image_id][1].astype(np.uint8)

        n0 = ndi.label(seed)[1]

        h, w = seed.shape
        nh = max(1, int(round(h * s)))
        nw = max(1, int(round(w * s)))

        # What Stage 10 effectively uses
        nn = cv2.resize(
            seed,
            (nw, nh),
            interpolation=cv2.INTER_NEAREST,
        ).astype(np.uint8)

        # Seed-safe alternative
        ar_float = cv2.resize(
            seed.astype(np.float32),
            (nw, nh),
            interpolation=cv2.INTER_AREA,
        )
        ar = (ar_float > 0).astype(np.uint8)

        base_cc += n0
        nn_cc += ndi.label(nn)[1]
        area_cc += ndi.label(ar)[1]

        base_px += int(seed.sum())
        nn_px += int(nn.sum())
        area_px += int(ar.sum())

    print(
        f"scale {s:.2f} | "
        f"CC kept nearest {nn_cc/base_cc:6.1%} | "
        f"area>0 {area_cc/base_cc:6.1%} | "
        f"seed px nearest {nn_px/max(base_px,1):6.1%} | "
        f"area>0 {area_px/max(base_px,1):6.1%}"
    )

print(
    "\nInterpretation:\n"
    "- If nearest connected-component retention drops noticeably at 0.70/0.68,\n"
    "  small erosion cores are being lost.\n"
    "- area>0 is intentionally conservative: any seed evidence in a resize cell survives.\n"
    "- Do NOT expect seed pixel totals to remain 100%, because the image itself is smaller."
)

building erosion-0.70 targets ...
targets (erosion 0.7) from cache
  seed channel covers 1.56% of pixels at erosion 0.7
testing 40 training images

scale 0.85 | CC kept nearest 101.4% | area>0  75.8% | seed px nearest  72.2% | area>0 111.7%
scale 0.80 | CC kept nearest 101.8% | area>0  75.8% | seed px nearest  63.9% | area>0  98.3%
scale 0.75 | CC kept nearest 103.2% | area>0  77.6% | seed px nearest  56.2% | area>0  82.8%
scale 0.70 | CC kept nearest 102.4% | area>0  75.2% | seed px nearest  49.0% | area>0  81.1%
scale 0.68 | CC kept nearest 102.1% | area>0  75.0% | seed px nearest  46.3% | area>0  79.2%

Interpretation:
- If nearest connected-component retention drops noticeably at 0.70/0.68,
  small erosion cores are being lost.
- area>0 is intentionally conservative: any seed evidence in a resize cell survives.
- Do NOT expect seed pixel totals to remain 100%, because the image itself is smaller.


In [20]:
# ── C · STAGE 11 DENSITY AUGMENTATION PATCH ──────────────────────────────────

# Keep Stage 10's best controlled seed target
STAGE11_EROSION = 0.70

# New augmentation bundle
CFG.AUG_D4 = True

# Density-biased scale mixture
CFG.AUG_SCALE_MIX = [
    (0.45, 0.68, 0.82),   # dense / zoomed-out mode
    (0.35, 0.85, 1.00),   # normal mode
    (0.20, 1.00, 1.20),   # zoom-in robustness
]

# Dense-image oversampling
CFG.DENSE_GT_THR = 200
CFG.DENSE_REPEAT = 2


def sample_scale_stage11():
    """
    Sample from the Stage 11 scale mixture.

    45% of samples deliberately zoom out enough to expose
    much larger portions of dense microscopy fields.
    """
    r = random.random()
    acc = 0.0

    for p, lo, hi in CFG.AUG_SCALE_MIX:
        acc += p
        if r < acc:
            return random.uniform(lo, hi)

    # Safety fallback
    _, lo, hi = CFG.AUG_SCALE_MIX[-1]
    return random.uniform(lo, hi)


def resize_pair_stage11(img, tgt, scale):
    """
    Resize the FULL image and targets before the dataset crop.

    Cell B showed:
      - INTER_AREA > 0 on seed masks merged too many seed components.
      - INTER_NEAREST kept seed connected-component counts close to original.

    Therefore:
      image       -> INTER_LINEAR
      foreground  -> INTER_AREA + >0.5 when downscaling
      seed/others -> INTER_NEAREST
    """

    h, w = img.shape

    nh = max(64, int(round(h * scale)))
    nw = max(64, int(round(w * scale)))

    # Image interpolation
    img_r = cv2.resize(
        img,
        (nw, nh),
        interpolation=cv2.INTER_LINEAR,
    ).astype(np.float32)

    chans = []

    for c in range(tgt.shape[0]):

        # Foreground channel
        if c == 0 and scale < 1.0:
            fg = cv2.resize(
                tgt[c].astype(np.float32),
                (nw, nh),
                interpolation=cv2.INTER_AREA,
            )

            out = (fg > 0.5).astype(np.uint8)

        else:
            # Seed / border / any remaining discrete target channels
            out = cv2.resize(
                tgt[c].astype(np.uint8),
                (nw, nh),
                interpolation=cv2.INTER_NEAREST,
            ).astype(np.uint8)

        chans.append(out)

    tgt_r = np.stack(chans, axis=0)

    return img_r, tgt_r


def _augment_stage11(self, img, tgt):
    """
    Stage 11 augmentation:
      1. D4 rotation
      2. H/V flips
      3. FULL-FRAME density-biased scaling
      4. existing mild brightness / contrast / noise

    Cropping still happens afterward inside CellDataset.
    """

    # ── D4 rotation ───────────────────────────────────────────────────────────
    if CFG.AUG_D4:
        k = random.randint(0, 3)

        if k:
            img = np.rot90(img, k)
            tgt = np.rot90(tgt, k, axes=(1, 2))

    # ── Existing flips ───────────────────────────────────────────────────────
    if random.random() < CFG.AUG_HFLIP:
        img = img[:, ::-1]
        tgt = tgt[:, :, ::-1]

    if random.random() < CFG.AUG_VFLIP:
        img = img[::-1, :]
        tgt = tgt[:, ::-1, :]

    img = np.ascontiguousarray(img)
    tgt = np.ascontiguousarray(tgt)

    # ── KEY STAGE 11 CHANGE ──────────────────────────────────────────────────
    # Scale the FULL image BEFORE the fixed crop.
    #
    # At ~0.70:
    # 520×704 -> ~364×493
    #
    # So essentially an entire dense microscopy field can fit inside
    # the 400×512 training crop.
    s = sample_scale_stage11()

    if abs(s - 1.0) > 1e-3:
        img, tgt = resize_pair_stage11(img, tgt, s)

    # ── Existing mild photometric augmentation ───────────────────────────────

    # brightness
    img = (
        img
        + random.uniform(
            -CFG.AUG_BRIGHT,
            CFG.AUG_BRIGHT,
        ) * STD
    )

    # contrast
    contrast = (
        1.0
        + random.uniform(
            -CFG.AUG_CONTRAST,
            CFG.AUG_CONTRAST,
        )
    )

    img = (img - MEAN) * contrast + MEAN

    # Gaussian noise
    img = (
        img
        + np.random.randn(*img.shape).astype(np.float32)
        * CFG.AUG_NOISE
        * STD
    )

    img = np.clip(
        img,
        0.0,
        1.0,
    ).astype(np.float32)

    tgt = tgt.astype(np.uint8)

    return img, tgt


# ── Install Stage 11 augmentation into the LIVE Stage 10 Dataset class ────────

CellDataset.augment = _augment_stage11

print(">>> Stage 11 augmentation installed")


# ── Dense-image oversampling ──────────────────────────────────────────────────

GT_COUNT = {
    image_id: len(INST[image_id])
    for image_id in train_ids
}

dense_ids = [
    image_id
    for image_id in train_ids
    if GT_COUNT[image_id] >= CFG.DENSE_GT_THR
]

train_ids_boost = (
    list(train_ids)
    + dense_ids * CFG.DENSE_REPEAT
)

dense_draws = (
    len(dense_ids)
    * (1 + CFG.DENSE_REPEAT)
)

dense_share = (
    dense_draws
    / len(train_ids_boost)
)

print(
    f"dense images >= {CFG.DENSE_GT_THR} GT: "
    f"{len(dense_ids)}/{len(train_ids)}"
)

print(
    f"training id list: "
    f"{len(train_ids)} -> {len(train_ids_boost)}"
)

print(
    f"approx dense sampling share after repeat: "
    f"{dense_share:.1%}"
)


# ── Sanity-check target generation is still erosion 0.70 ─────────────────────

load_targets(STAGE11_EROSION)

print(
    f">>> Stage 11 target erosion fixed at "
    f"{STAGE11_EROSION:.2f}"
)


# ── Dataset sanity test ────────────────────────────────────────────────────────

_ds = CellDataset(
    train_ids_boost,
    CFG.PROBE_CROP,
    True,
)

_x, _y, _w, _meta = _ds[0]

print(
    "sample shapes:",
    tuple(_x.shape),
    tuple(_y.shape),
)

assert tuple(_x.shape[-2:]) == tuple(CFG.PROBE_CROP), (
    f"image crop shape wrong: "
    f"{tuple(_x.shape[-2:])} != {CFG.PROBE_CROP}"
)

assert tuple(_y.shape[-2:]) == tuple(CFG.PROBE_CROP), (
    f"target crop shape wrong: "
    f"{tuple(_y.shape[-2:])} != {CFG.PROBE_CROP}"
)

assert _y.shape[0] >= 2, (
    "Expected at least foreground + seed target channels."
)

print(
    "sample target pixels:",
    [int(_y[c].sum()) for c in range(_y.shape[0])]
)

del _ds, _x, _y, _w, _meta
gc.collect()

print(">>> Stage 11 dataset sanity check passed")

>>> Stage 11 augmentation installed
dense images >= 200 GT: 73/338
training id list: 338 -> 484
approx dense sampling share after repeat: 45.2%
>>> Stage 11 target erosion fixed at 0.70
sample shapes: (3, 400, 512) (2, 400, 512)
sample target pixels: [40257, 5009]
>>> Stage 11 dataset sanity check passed


In [24]:
# ── D · STAGE 11 TRAINING — EXACT STAGE 10 API, NEW ARTIFACTS ONLY ────────────

STAGE11_RUN_TAG = "stage11_density_aug"

STAGE11_SPEC = {
    "name": "aug70",
    "erosion": 0.70,
    "seed": 43,
}

# ── Preflight: fail here BEFORE training if the live session is missing anything
required_stage11 = [
    "UNet",
    "loss_fn",
    "CellDataset",
    "train_ids_boost",
    "quick_val_f1",
    "predict_maps",
    "maps_to_float",
    "write_submission",
    "load_targets",
    "INST",
    "val_ids",
    "TEST_IDS",
    "TEST_DIR",
    "CKPT",
    "MAPS",
    "ART",
    "CFG",
]

missing = [name for name in required_stage11 if name not in globals()]

assert not missing, (
    "Stage 11 is missing live Stage 10 objects: "
    + ", ".join(missing)
)

print(">>> Stage 11 preflight passed")
print("model class: UNet")
print("loss: loss_fn")
print("training IDs:", len(train_ids_boost))
print("erosion:", STAGE11_SPEC["erosion"])
print("seed:", STAGE11_SPEC["seed"])
print("crop:", CFG.PROBE_CROP)


def train_probe_stage11(spec):
    """
    Stage 11 = Stage 10 training protocol with ONLY:
      - Cell C's density-targeted augmentation
      - train_ids_boost dense-image oversampling

    Stage 10 artifacts are never overwritten.
    """

    member = spec["name"]
    erosion = float(spec["erosion"])
    seed = int(spec["seed"])

    tag = f"{STAGE11_RUN_TAG}_{member}"

    best_path = CKPT / f"{tag}_best.pt"
    val_map_path = MAPS / f"{tag}_val.npz"
    test_map_path = MAPS / f"{tag}_test.npz"
    log_path = ART / f"{tag}_train_log.csv"

    print("\n" + "=" * 72)
    print("STAGE 11 MEMBER:", member)
    print("tag:", tag)
    print("erosion:", erosion)
    print("seed:", seed)
    print("crop:", CFG.PROBE_CROP)
    print("training IDs:", len(train_ids_boost))
    print("=" * 72)

    # -------------------------------------------------------------------------
    # 1. Exact same erosion target as Stage 10 e70
    # -------------------------------------------------------------------------
    load_targets(erosion)

    # Stage 11 is controlled against Stage 10 e70
    seed_everything(seed)

    # -------------------------------------------------------------------------
    # 2. Exact Stage 10 architecture
    # -------------------------------------------------------------------------
    model = UNet(
        CFG.ENCODER,
        CFG.OUT_CHANNELS,
        CFG.DECODER_CHANNELS,
        pretrained=True,
    ).to(DEVICE)

    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}

    dec_params = [
        p for p in model.parameters()
        if id(p) not in enc_ids
    ]

    # Exact Stage 10 optimizer
    opt = torch.optim.AdamW(
        [
            {
                "params": dec_params,
                "lr": CFG.SUP_LR,
            },
            {
                "params": enc_params,
                "lr": CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT,
            },
        ],
        weight_decay=CFG.SUP_WD,
    )

    # -------------------------------------------------------------------------
    # 3. ONLY Stage 11 loader difference: boosted dense image list
    # -------------------------------------------------------------------------
    loader = DataLoader(
        CellDataset(
            train_ids_boost,
            CFG.PROBE_CROP,
            True,
        ),
        batch_size=CFG.SUP_BATCH,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
    )

    total_steps = CFG.SUP_EPOCHS * CFG.STEPS_PER_EPOCH

    scaler = make_scaler()

    best_f1 = -1.0
    best_epoch = -1
    history = []

    t0 = time.time()

    # Exact Stage 10 cosine/warmup schedule
    def lr_at(step):
        if step < CFG.WARMUP_STEPS:
            return (step + 1) / max(1, CFG.WARMUP_STEPS)

        p = (
            (step - CFG.WARMUP_STEPS)
            / max(1, total_steps - CFG.WARMUP_STEPS)
        )

        return 0.5 * (
            1.0
            + math.cos(
                math.pi * min(1.0, p)
            )
        )

    step = 0
    it = iter(loader)

    # -------------------------------------------------------------------------
    # 4. Train — copied from Stage 10 protocol
    # -------------------------------------------------------------------------
    for ep in range(CFG.SUP_EPOCHS):

        freeze = ep < CFG.FREEZE_ENCODER_EPOCHS

        for p in enc_params:
            p.requires_grad_(not freeze)

        model.train()

        run_loss = 0.0

        for _ in range(CFG.STEPS_PER_EPOCH):

            try:
                x, y, w, _ = next(it)

            except StopIteration:
                it = iter(loader)
                x, y, w, _ = next(it)

            x = normalize_batch(x).to(
                DEVICE,
                non_blocking=True,
            )

            y = y.to(
                DEVICE,
                non_blocking=True,
            )

            w = w.to(
                DEVICE,
                non_blocking=True,
            )

            mult = lr_at(step)

            # decoder LR
            opt.param_groups[0]["lr"] = (
                CFG.SUP_LR * mult
            )

            # encoder LR, zero while frozen
            opt.param_groups[1]["lr"] = (
                CFG.SUP_LR
                * CFG.SUP_ENCODER_LR_MULT
                * mult
                * (0.0 if freeze else 1.0)
            )

            opt.zero_grad(set_to_none=True)

            with amp_autocast():
                logits = model(x)
                loss = loss_fn(
                    logits,
                    y,
                    w,
                )

            scaler.scale(loss).backward()

            scaler.unscale_(opt)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CFG.GRAD_CLIP,
            )

            scaler.step(opt)
            scaler.update()

            run_loss += float(loss.item())
            step += 1

        epoch_loss = (
            run_loss
            / CFG.STEPS_PER_EPOCH
        )

        msg = (
            f"[{member}] "
            f"ep {ep + 1:02d}/{CFG.SUP_EPOCHS} "
            f"loss {epoch_loss:.4f}"
        )

        # ---------------------------------------------------------------------
        # Same validation timing and same Stage 10 DEFAULT_POST
        # ---------------------------------------------------------------------
        if (
            ep + 1 >= CFG.EVAL_START
            and
            (ep + 1 - CFG.EVAL_START)
            % CFG.EVAL_EVERY == 0
        ):

            f1 = quick_val_f1(
                model,
                val_ids,
                CFG.DEFAULT_POST,
            )

            msg += f" | val F1 {f1:.4f}"

            history.append(
                {
                    "member": member,
                    "erosion": erosion,
                    "epoch": ep + 1,
                    "loss": epoch_loss,
                    "f1": f1,
                }
            )

            if f1 > best_f1:

                best_f1 = f1
                best_epoch = ep + 1

                torch.save(
                    {
                        "run_tag": STAGE11_RUN_TAG,
                        "member": member,
                        "epoch": best_epoch,
                        "f1": best_f1,
                        "state_dict": model.state_dict(),
                        "spec": dict(spec),
                        "encoder": CFG.ENCODER,
                    },
                    best_path,
                )

                msg += "  *"

        print(
            msg
            + f" | {fmt(time.time() - t0)}",
            flush=True,
        )

    # Safety: normally impossible because evaluation starts at epoch 24
    if best_f1 < 0:

        best_f1 = quick_val_f1(
            model,
            val_ids,
            CFG.DEFAULT_POST,
        )

        best_epoch = CFG.SUP_EPOCHS

        torch.save(
            {
                "run_tag": STAGE11_RUN_TAG,
                "member": member,
                "epoch": best_epoch,
                "f1": best_f1,
                "state_dict": model.state_dict(),
                "spec": dict(spec),
                "encoder": CFG.ENCODER,
            },
            best_path,
        )

    pd.DataFrame(history).to_csv(
        log_path,
        index=False,
    )

    print(
        f"\n[{member}] BEST epoch {best_epoch} "
        f"| quick F1 {best_f1:.4f}"
    )

    print(
        f"[{member}] checkpoint: {best_path}"
    )

    # -------------------------------------------------------------------------
    # 5. Reload the BEST Stage 11 checkpoint
    # -------------------------------------------------------------------------
    blob = torch.load(
        best_path,
        map_location="cpu",
    )

    assert blob["run_tag"] == STAGE11_RUN_TAG
    assert blob["member"] == member

    best_model = UNet(
        CFG.ENCODER,
        CFG.OUT_CHANNELS,
        CFG.DECODER_CHANNELS,
        pretrained=False,
    ).to(DEVICE)

    best_model.load_state_dict(
        blob["state_dict"]
    )

    best_model.eval()

    # Training model no longer needed
    del model, opt, loader, scaler
    gc.collect()
    torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # 6. Generate Stage 11 TTA maps using EXACT Stage 10 inference API
    # -------------------------------------------------------------------------
    print(
        f"\n[{member}] generating validation TTA maps..."
    )

    val_maps_u8 = predict_maps(
        best_model,
        val_ids,
        None,
    )

    np.savez_compressed(
        val_map_path,
        **val_maps_u8,
    )

    print(
        f"[{member}] saved:",
        val_map_path,
    )

    print(
        f"\n[{member}] generating test TTA maps..."
    )

    test_maps_u8 = predict_maps(
        best_model,
        TEST_IDS,
        TEST_DIR,
    )

    np.savez_compressed(
        test_map_path,
        **test_maps_u8,
    )

    print(
        f"[{member}] saved:",
        test_map_path,
    )

    # -------------------------------------------------------------------------
    # 7. Direct Stage-10-style TTA comparison using DEFAULT_POST
    # -------------------------------------------------------------------------
    tta_vals = []

    for image_id in val_ids:

        fg, seed_map = maps_to_float(
            val_maps_u8[image_id]
        )

        prep = prepare_eval(
            fg,
            seed_map,
            INST[image_id],
            CFG.DEFAULT_POST["t_cell"],
            CFG.DEFAULT_POST["t_seed"],
        )

        score = f1_filtered(
            prep,
            CFG.DEFAULT_POST["min_area"],
            CFG.DEFAULT_POST["score_thr"],
        )

        tta_vals.append(score)

    tta_f1 = float(
        np.mean(tta_vals)
    )

    # -------------------------------------------------------------------------
    # 8. Test count using same DEFAULT_POST
    # -------------------------------------------------------------------------
    test_counts = []

    for image_id in TEST_IDS:

        fg, seed_map = maps_to_float(
            test_maps_u8[image_id]
        )

        pred_inst = instances_from_maps(
            fg,
            seed_map,
            CFG.DEFAULT_POST,
        )

        test_counts.append(
            len(pred_inst)
        )

    mean_test_pred = float(
        np.mean(test_counts)
    )

    print("\n" + "=" * 72)

    print(
        f"STAGE 11 {member} COMPLETE"
    )

    print(
        f"quick checkpoint F1: {best_f1:.4f}"
    )

    print(
        f"TTA F1 @ Stage10 default post: "
        f"{tta_f1:.4f}"
    )

    print(
        f"test predictions/image: "
        f"{mean_test_pred:.1f}"
    )

    print(
        f"checkpoint: {best_path}"
    )

    print("=" * 72)

    del best_model, blob
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "member": member,
        "erosion": erosion,
        "quick_f1": best_f1,
        "best_epoch": best_epoch,
        "tta_f1": tta_f1,
        "mean": mean_test_pred,
        "ckpt": best_path,
        "val_maps": val_map_path,
        "test_maps": test_map_path,
        "log": log_path,
    }


print("\n>>> Stage 11 training function ready")
print(">>> THIS USES THE ACTUAL STAGE 10 UNet + loss_fn PIPELINE")
print(">>> Stage 10 artifacts will NOT be overwritten")
print("spec:", STAGE11_SPEC)

>>> Stage 11 preflight passed
model class: UNet
loss: loss_fn
training IDs: 484
erosion: 0.7
seed: 43
crop: (400, 512)

>>> Stage 11 training function ready
>>> THIS USES THE ACTUAL STAGE 10 UNet + loss_fn PIPELINE
>>> Stage 10 artifacts will NOT be overwritten
spec: {'name': 'aug70', 'erosion': 0.7, 'seed': 43}


In [26]:
# ── FIX GLOBAL TORCH.NN ALIAS AFTER CELL B ───────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F

# Verify the aliases are correct before starting an expensive run
assert nn is torch.nn
assert hasattr(nn, "Sequential")
assert hasattr(nn, "Conv2d")
assert hasattr(nn, "BatchNorm2d")

print(">>> torch.nn restored correctly")
print("nn.Sequential:", nn.Sequential)
print("nn.Conv2d:", nn.Conv2d)

>>> torch.nn restored correctly
nn.Sequential: <class 'torch.nn.modules.container.Sequential'>
nn.Conv2d: <class 'torch.nn.modules.conv.Conv2d'>


In [27]:
# ── E · RUN STAGE 11 ─────────────────────────────────────────────────────────

STAGE11_RESULT = train_probe_stage11(STAGE11_SPEC)

print("\n=== STAGE 11 RESULT ===")
print(STAGE11_RESULT)


STAGE 11 MEMBER: aug70
tag: stage11_density_aug_aug70
erosion: 0.7
seed: 43
crop: (400, 512)
training IDs: 484
[aug70] ep 01/70 loss 1.5223 | 0h 00m 09s
[aug70] ep 02/70 loss 1.3716 | 0h 00m 19s
[aug70] ep 03/70 loss 1.2786 | 0h 00m 33s
[aug70] ep 04/70 loss 1.2008 | 0h 00m 47s
[aug70] ep 05/70 loss 1.1458 | 0h 01m 00s
[aug70] ep 06/70 loss 1.0851 | 0h 01m 14s
[aug70] ep 07/70 loss 1.0460 | 0h 01m 27s
[aug70] ep 08/70 loss 0.9844 | 0h 01m 41s
[aug70] ep 09/70 loss 0.9577 | 0h 01m 55s
[aug70] ep 10/70 loss 0.9179 | 0h 02m 09s
[aug70] ep 11/70 loss 0.8812 | 0h 02m 22s
[aug70] ep 12/70 loss 0.8447 | 0h 02m 36s
[aug70] ep 13/70 loss 0.8190 | 0h 02m 49s
[aug70] ep 14/70 loss 0.7866 | 0h 03m 03s
[aug70] ep 15/70 loss 0.7576 | 0h 03m 17s
[aug70] ep 16/70 loss 0.7317 | 0h 03m 31s
[aug70] ep 17/70 loss 0.7107 | 0h 03m 44s
[aug70] ep 18/70 loss 0.6904 | 0h 03m 58s
[aug70] ep 19/70 loss 0.6601 | 0h 04m 12s
[aug70] ep 20/70 loss 0.6402 | 0h 04m 25s
[aug70] ep 21/70 loss 0.6188 | 0h 04m 39s
[aug70

In [28]:
# ── F · STAGE 11 FINAL SWEEP + 11th_submission.csv ───────────────────────────

# Load Stage 11 validation/test maps
with np.load(STAGE11_RESULT["val_maps"]) as z:
    STAGE11_VAL_MAPS = {
        k: z[k].astype(np.float32) / 255.0
        for k in z.files
    }

with np.load(STAGE11_RESULT["test_maps"]) as z:
    STAGE11_TEST_MAPS = {
        k: z[k].astype(np.float32) / 255.0
        for k in z.files
    }

print(
    "loaded Stage 11 maps:",
    len(STAGE11_VAL_MAPS), "val |",
    len(STAGE11_TEST_MAPS), "test"
)


# ── Tune Stage 11 post-processing on all 86 validation images ────────────────

TC_GRID = [0.40, 0.45, 0.50, 0.55, 0.60]
TS_GRID = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
MA_GRID = [30, 50, 70, 90, 110]

rows11 = []

for tc in TC_GRID:
    for ts in TS_GRID:

        preps = [
            prepare_eval(
                STAGE11_VAL_MAPS[i][0],
                STAGE11_VAL_MAPS[i][1],
                INST[i],
                tc,
                ts,
            )
            for i in val_ids
        ]

        for ma in MA_GRID:
            vals = np.array(
                [
                    f1_filtered(
                        p,
                        ma,
                        0.0,
                    )
                    for p in preps
                ]
            )

            mean_pred = np.mean(
                [
                    int(
                        (
                            (p["area"][1:] >= ma)
                            & (p["score"][1:] >= 0.0)
                        ).sum()
                    )
                    for p in preps
                ]
            )

            rows11.append(
                {
                    "t_cell": tc,
                    "t_seed": ts,
                    "min_area": ma,
                    "score_thr": 0.0,
                    "f1": float(vals.mean()),
                    "mean_pred": float(mean_pred),
                }
            )

        del preps

STAGE11_SWEEP = (
    pd.DataFrame(rows11)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("\n=== STAGE 11 TOP POST-PROCESSING ===")
print(STAGE11_SWEEP.head(10).to_string(index=False))


# Plateau-centre selection, same robustness idea as Stage 10
peak11 = float(STAGE11_SWEEP.f1.max())

plateau11 = STAGE11_SWEEP[
    STAGE11_SWEEP.f1 >= peak11 - 0.004
]

def snap11(value, grid):
    return min(grid, key=lambda x: abs(x - value))

STAGE11_POST = {
    "t_cell": snap11(
        float(np.median(plateau11.t_cell)),
        TC_GRID,
    ),
    "t_seed": snap11(
        float(np.median(plateau11.t_seed)),
        TS_GRID,
    ),
    "min_area": int(
        snap11(
            int(np.median(plateau11.min_area)),
            MA_GRID,
        )
    ),
    "score_thr": 0.0,
}

# Evaluate selected plateau centre
vals11 = []

for image_id in val_ids:
    prep = prepare_eval(
        STAGE11_VAL_MAPS[image_id][0],
        STAGE11_VAL_MAPS[image_id][1],
        INST[image_id],
        STAGE11_POST["t_cell"],
        STAGE11_POST["t_seed"],
    )

    vals11.append(
        f1_filtered(
            prep,
            STAGE11_POST["min_area"],
            STAGE11_POST["score_thr"],
        )
    )

STAGE11_LOCAL_F1 = float(np.mean(vals11))

print(
    f"\nStage 11 peak F1: {peak11:.4f}"
)

print(
    f"Stage 11 plateau post: {STAGE11_POST}"
)

print(
    f"Stage 11 selected local F1: {STAGE11_LOCAL_F1:.4f}"
)


# ── Write FINAL Stage 11 submission ──────────────────────────────────────────

STAGE11_SUBMISSION_PATH = WORK / "11th_submission.csv"

rows = []
counts = []
fallbacks = 0

for image_id in TEST_IDS:

    fg = STAGE11_TEST_MAPS[image_id][0]
    seed = STAGE11_TEST_MAPS[image_id][1]

    inst = instances_from_maps(
        fg,
        seed,
        STAGE11_POST,
    )

    if not inst:
        fallbacks += 1

        inst = instances_from_maps(
            fg,
            seed,
            {
                "t_cell": 0.30,
                "t_seed": 0.30,
                "min_area": 0,
                "score_thr": 0.0,
            },
        )[:1]

    counts.append(len(inst))

    for flat in inst:
        rows.append(
            (
                image_id,
                flat_to_rle(flat),
            )
        )


submission11 = pd.DataFrame(
    rows,
    columns=[ID_COL, ANN_COL],
)

submission11.insert(
    0,
    CFG.SUB_INDEX_COL,
    np.arange(
        len(submission11),
        dtype=np.int64,
    ),
)

submission11.to_csv(
    STAGE11_SUBMISSION_PATH,
    index=False,
)

print("\n=== STAGE 11 SUBMISSION ===")
print("path:", STAGE11_SUBMISSION_PATH)
print("rows:", len(submission11))
print("mean predictions/image:", np.mean(counts))
print("min predictions/image:", np.min(counts))
print("max predictions/image:", np.max(counts))
print("fallbacks:", fallbacks)

print(
    "\n>>> READY TO SUBMIT:",
    STAGE11_SUBMISSION_PATH,
)

# ── Write Stage 11 report ────────────────────────────────────────────────────

STAGE11_REPORT_PATH = WORK / "11th_REPORT.md"

report11 = f"""# Stage 11 Report — Density-Targeted Augmentation

## Model
- Encoder: ConvNeXt-Tiny, ImageNet-21k pretrained
- Architecture: U-Net + foreground/seed prediction + watershed
- Seed erosion fraction: 0.70
- Seed: 43
- Crop: 400x512
- Member: aug70

## Stage 11 Changes
Stage 11 keeps the Stage 10 e70 model/training setup and changes the training-data exposure:

- D4 / 90-degree rotation augmentation
- Full-image scaling before cropping
- Density-biased scale mixture:
  - 45%: 0.68-0.82
  - 35%: 0.85-1.00
  - 20%: 1.00-1.20
- Dense-image oversampling
  - dense threshold: >= {CFG.DENSE_GT_THR} GT instances
  - repeat factor: {CFG.DENSE_REPEAT}
- Foreground downscaling uses INTER_AREA
- Seed/discrete targets use INTER_NEAREST

Dense images in training:
- {len(dense_ids)} / {len(train_ids)}
- boosted training list: {len(train_ids_boost)}
- approximate dense sampling share: {dense_share:.1%}

## Stage 10 Density Audit

The Stage 10 e70 audit showed that undercounting is concentrated in dense images:

| Density | Images | Mean F1 | Pred / GT |
|---|---:|---:|---:|
| Sparse <50 | 41 | 0.8573 | 0.9484 |
| Medium 50-199 | 28 | 0.7569 | 0.9860 |
| Dense 200-399 | 11 | 0.7792 | 0.9036 |
| Very dense >=400 | 6 | 0.7518 | 0.8703 |

This motivated Stage 11's density-targeted augmentation strategy.

## Validation Result
- Peak validation F1: {peak11:.4f}
- Selected plateau F1: {STAGE11_LOCAL_F1:.4f}
- Selected post-processing:
  - t_cell: {STAGE11_POST['t_cell']}
  - t_seed: {STAGE11_POST['t_seed']}
  - min_area: {STAGE11_POST['min_area']}
  - score_thr: {STAGE11_POST['score_thr']}

## Test Prediction Statistics
- Mean predictions/image: {np.mean(counts):.1f}
- Minimum: {np.min(counts)}
- Maximum: {np.max(counts)}
- Fallbacks: {fallbacks}

## Reference
Stage 10 e70:
- local plateau F1: 0.8072
- local peak F1: 0.8086
- public score: 0.5295

Stage 11 tests whether deliberately exposing the model to larger portions of dense microscopy
fields improves dense-image recall beyond the Stage 10 seed-target optimum.

## Output
- Submission: `/kaggle/working/11th_submission.csv`
- Report: `/kaggle/working/11th_REPORT.md`
"""

STAGE11_REPORT_PATH.write_text(report11)

print("\n=== STAGE 11 REPORT ===")
print("path:", STAGE11_REPORT_PATH)
print(report11)

loaded Stage 11 maps: 86 val | 182 test

=== STAGE 11 TOP POST-PROCESSING ===
 t_cell  t_seed  min_area  score_thr       f1  mean_pred
   0.45    0.50        90        0.0 0.801852 108.372093
   0.45    0.55        90        0.0 0.801498 107.860465
   0.40    0.50       110        0.0 0.801261 105.639535
   0.55    0.50        90        0.0 0.801249 106.290698
   0.40    0.50        90        0.0 0.801228 109.162791
   0.50    0.55        90        0.0 0.801185 107.127907
   0.55    0.55        90        0.0 0.800971 106.023256
   0.50    0.50        90        0.0 0.800858 107.476744
   0.55    0.50        70        0.0 0.800788 110.127907
   0.45    0.50        70        0.0 0.800539 111.686047

Stage 11 peak F1: 0.8019
Stage 11 plateau post: {'t_cell': 0.5, 't_seed': 0.55, 'min_area': 90, 'score_thr': 0.0}
Stage 11 selected local F1: 0.8012

=== STAGE 11 SUBMISSION ===
path: /kaggle/working/11th_submission.csv
rows: 19245
mean predictions/image: 105.74175824175825
min predictions/ima

## Stage 12

In [30]:
# ── STAGE 12 PREFLIGHT + ROBUST E70 MAP RELOAD ────────────────────────────────
# Based on the actual live Stage 10/11 notebook.
#
# Stage 10 deliberately deleted VAL_BEST / TEST_BEST after submission.
# Cell A later restored VAL_BEST only.
# This cell safely reconstructs BOTH from the cached e70 maps.

import gc
import numpy as np

# ------------------------------------------------------------------
# 1. Verify everything that must already exist in the live session
# ------------------------------------------------------------------

required = [
    "MAPS",
    "CFG",
    "val_ids",
    "TEST_IDS",
    "INST",
    "prepare_eval",
    "f1_filtered",
    "STAGE11_VAL_MAPS",
    "STAGE11_TEST_MAPS",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required live objects: "
    + ", ".join(missing)
)


# ------------------------------------------------------------------
# 2. Locate the actual Stage 10 e70 cached maps
# ------------------------------------------------------------------

e70_val_candidates = [
    MAPS / f"{CFG.RUN_TAG}_e70_val.npz",
    MAPS / "stage10_erosion_e70_val.npz",
]

e70_test_candidates = [
    MAPS / f"{CFG.RUN_TAG}_e70_test.npz",
    MAPS / "stage10_erosion_e70_test.npz",
]


E70_VAL_PATH = next(
    (
        p
        for p in e70_val_candidates
        if p.exists()
    ),
    None,
)

E70_TEST_PATH = next(
    (
        p
        for p in e70_test_candidates
        if p.exists()
    ),
    None,
)


# Fallback search only if exact Stage 10 filenames are unavailable.
if E70_VAL_PATH is None:

    found = sorted(
        MAPS.glob("*e70*val*.npz")
    )

    assert found, (
        f"Could not find Stage 10 e70 validation maps in {MAPS}"
    )

    E70_VAL_PATH = found[0]


if E70_TEST_PATH is None:

    found = sorted(
        MAPS.glob("*e70*test*.npz")
    )

    assert found, (
        f"Could not find Stage 10 e70 test maps in {MAPS}"
    )

    E70_TEST_PATH = found[0]


print("Stage 10 e70 val maps :", E70_VAL_PATH)
print("Stage 10 e70 test maps:", E70_TEST_PATH)


# ------------------------------------------------------------------
# 3. Load NPZ maps into float [0, 1]
# ------------------------------------------------------------------

def load_float_maps(path, expected_ids):

    with np.load(path) as z:

        available = set(z.files)
        expected = set(expected_ids)

        missing_ids = expected - available
        extra_ids = available - expected

        assert not missing_ids, (
            f"{path.name} is missing "
            f"{len(missing_ids)} expected images: "
            f"{sorted(missing_ids)[:5]}"
        )

        if extra_ids:
            print(
                f"note: ignoring {len(extra_ids)} extra IDs "
                f"in {path.name}"
            )

        result = {}

        for image_id in expected_ids:

            arr = z[image_id]

            assert arr.ndim == 3, (
                f"{image_id}: expected CHW map, "
                f"got shape {arr.shape}"
            )

            assert arr.shape[0] == 2, (
                f"{image_id}: expected 2 channels "
                f"(foreground + seed), got {arr.shape}"
            )

            # Stage 10 cached maps are uint8 [0,255].
            if arr.dtype == np.uint8:
                arr = (
                    arr.astype(np.float32)
                    / 255.0
                )

            else:
                arr = arr.astype(np.float32)

                # Safety for any non-uint8 map accidentally
                # stored in 0..255 range.
                if arr.max() > 1.5:
                    arr /= 255.0

            assert np.isfinite(arr).all(), (
                f"{image_id}: map contains NaN/Inf"
            )

            assert (
                arr.min() >= -1e-6
                and arr.max() <= 1.0001
            ), (
                f"{image_id}: probability range invalid: "
                f"{arr.min()} .. {arr.max()}"
            )

            result[image_id] = arr

    return result


E70_VAL = load_float_maps(
    E70_VAL_PATH,
    val_ids,
)

E70_TEST = load_float_maps(
    E70_TEST_PATH,
    TEST_IDS,
)


# ------------------------------------------------------------------
# 4. Verify Stage 11 maps too
# ------------------------------------------------------------------

assert set(STAGE11_VAL_MAPS.keys()) == set(val_ids), (
    "Stage 11 validation-map IDs do not match val_ids."
)

assert set(STAGE11_TEST_MAPS.keys()) == set(TEST_IDS), (
    "Stage 11 test-map IDs do not match TEST_IDS."
)


for image_id in val_ids:

    assert (
        E70_VAL[image_id].shape
        == STAGE11_VAL_MAPS[image_id].shape
    ), (
        f"Validation shape mismatch for {image_id}: "
        f"e70={E70_VAL[image_id].shape}, "
        f"aug70={STAGE11_VAL_MAPS[image_id].shape}"
    )


for image_id in TEST_IDS:

    assert (
        E70_TEST[image_id].shape
        == STAGE11_TEST_MAPS[image_id].shape
    ), (
        f"Test shape mismatch for {image_id}: "
        f"e70={E70_TEST[image_id].shape}, "
        f"aug70={STAGE11_TEST_MAPS[image_id].shape}"
    )


# ------------------------------------------------------------------
# 5. Restore the names expected by Stage 12 Cells G / H
# ------------------------------------------------------------------

VAL_BEST = E70_VAL
TEST_BEST = E70_TEST

# Force POST to the actual Stage 10 e70 final plateau settings,
# not anything from Stage 11.
POST = {
    "t_cell": 0.50,
    "t_seed": 0.65,
    "min_area": 90,
    "score_thr": 0.0,
}


# ------------------------------------------------------------------
# 6. Final checks
# ------------------------------------------------------------------

assert len(VAL_BEST) == 86, (
    f"Expected 86 Stage 10 validation maps, got {len(VAL_BEST)}"
)

assert len(TEST_BEST) == 182, (
    f"Expected 182 Stage 10 test maps, got {len(TEST_BEST)}"
)

assert len(STAGE11_VAL_MAPS) == 86
assert len(STAGE11_TEST_MAPS) == 182


sample_id = val_ids[0]

print("\n>>> STAGE 12 PREFLIGHT PASSED")
print()
print("Stage 10 e70:")
print("  val maps :", len(VAL_BEST))
print("  test maps:", len(TEST_BEST))
print("  sample   :", VAL_BEST[sample_id].shape)
print(
    "  range    :",
    float(VAL_BEST[sample_id].min()),
    "to",
    float(VAL_BEST[sample_id].max()),
)

print()
print("Stage 11 aug70:")
print("  val maps :", len(STAGE11_VAL_MAPS))
print("  test maps:", len(STAGE11_TEST_MAPS))
print("  sample   :", STAGE11_VAL_MAPS[sample_id].shape)

print()
print("Stage 10 POST:", POST)
print()
print(">>> Cell G can now run unchanged.")

gc.collect()

Stage 10 e70 val maps : /kaggle/working/maps/stage10_erosion_e70_val.npz
Stage 10 e70 test maps: /kaggle/working/maps/stage10_erosion_e70_test.npz

>>> STAGE 12 PREFLIGHT PASSED

Stage 10 e70:
  val maps : 86
  test maps: 182
  sample   : (2, 520, 704)
  range    : 0.0 to 0.9960784316062927

Stage 11 aug70:
  val maps : 86
  test maps: 182
  sample   : (2, 520, 704)

Stage 10 POST: {'t_cell': 0.5, 't_seed': 0.65, 'min_area': 90, 'score_thr': 0.0}

>>> Cell G can now run unchanged.


3752

In [31]:
# ── G · e70 + aug70 blend ─────────────────────────────────────────────────────
# e70 float maps. If VAL_BEST/TEST_BEST are still e70's, this is a no-op rename.
E70_VAL, E70_TEST = VAL_BEST, TEST_BEST
A70_VAL, A70_TEST = STAGE11_VAL_MAPS, STAGE11_TEST_MAPS
assert set(E70_VAL) == set(A70_VAL) == set(val_ids), "map/id mismatch"

TC = [0.40, 0.45, 0.50, 0.55, 0.60]
TS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
MA = [30, 50, 70, 90, 110]
WS = [0.0, 0.25, 0.35, 0.50, 0.65, 0.75, 1.0]     # w = weight on aug70

rows = []
for w in WS:
    blend = {i: (1 - w) * E70_VAL[i] + w * A70_VAL[i] for i in val_ids}
    for tc in TC:
        for ts in TS:
            preps = [prepare_eval(blend[i][0], blend[i][1], INST[i], tc, ts) for i in val_ids]
            for ma in MA:
                f1 = float(np.mean([f1_filtered(p, ma, 0.0) for p in preps]))
                rows.append(dict(w=w, t_cell=tc, t_seed=ts, min_area=ma, f1=f1))
            del preps
    del blend
    print(f"w={w:.2f} done", flush=True)

BLEND = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
print(BLEND.head(12).to_string(index=False))

# Is the weight axis flat? Stage 9's was, and that was the tell.
print("\nbest F1 per weight:")
print(BLEND.groupby("w").f1.max().round(4).to_string())

peak = float(BLEND.f1.max())
plat = BLEND[BLEND.f1 >= peak - 0.004]
snap = lambda v, g: min(g, key=lambda x: abs(x - v))
BPOST = dict(w=snap(float(np.median(plat.w)), WS),
             t_cell=snap(float(np.median(plat.t_cell)), TC),
             t_seed=snap(float(np.median(plat.t_seed)), TS),
             min_area=int(snap(int(np.median(plat.min_area)), MA)), score_thr=0.0)
print("\nplateau centre:", BPOST)

wv = BPOST["w"]
bl = {i: (1 - wv) * E70_VAL[i] + wv * A70_VAL[i] for i in val_ids}
sel = float(np.mean([f1_filtered(
    prepare_eval(bl[i][0], bl[i][1], INST[i], BPOST["t_cell"], BPOST["t_seed"]),
    BPOST["min_area"], 0.0) for i in val_ids]))
print(f"selected blend local F1 {sel:.4f} | e70 reference 0.8072 | delta {sel - 0.8072:+.4f}")

w=0.00 done
w=0.25 done
w=0.35 done
w=0.50 done
w=0.65 done
w=0.75 done
w=1.00 done
   w  t_cell  t_seed  min_area       f1
0.35    0.50    0.55        90 0.811945
0.35    0.50    0.55       110 0.811169
0.35    0.50    0.60        90 0.810772
0.35    0.50    0.55        70 0.810740
0.50    0.55    0.50        90 0.810615
0.50    0.60    0.50        90 0.810365
0.35    0.50    0.50        90 0.810207
0.35    0.50    0.60        70 0.810108
0.35    0.55    0.55        90 0.810087
0.35    0.50    0.65        90 0.810069
0.35    0.50    0.50       110 0.809870
0.50    0.55    0.55        90 0.809860

best F1 per weight:
w
0.00    0.8086
0.25    0.8098
0.35    0.8119
0.50    0.8106
0.65    0.8084
0.75    0.8074
1.00    0.8019

plateau centre: {'w': 0.35, 't_cell': 0.5, 't_seed': 0.55, 'min_area': 90, 'score_thr': 0.0}
selected blend local F1 0.8119 | e70 reference 0.8072 | delta +0.0047


In [32]:
# ── H · how many misses are near-misses? ──────────────────────────────────────
def best_dice_per_gt(fg, seed, gt_flats, tc, ts, ma):
    lab, nlab = watershed_labels(fg, seed, tc, ts)
    if nlab == 0:
        return np.zeros(len(gt_flats))
    flat = lab.ravel()
    area = np.bincount(flat, minlength=nlab + 1).astype(np.float64)
    keep = area >= ma
    keep[0] = False
    out = []
    for idx in gt_flats:
        c = np.bincount(flat[idx], minlength=nlab + 1).astype(np.float64)
        d = 2.0 * c / np.maximum(idx.size + area, 1e-9)
        d[~keep] = 0.0
        out.append(float(d.max()))
    return np.array(out)

P = POST
band = {"hit >=0.50": 0, "near 0.35-0.50": 0, "weak 0.10-0.35": 0, "missed <0.10": 0}
per_bin = {}
for image_id in val_ids:
    n_gt = len(INST[image_id])
    b = ("sparse" if n_gt < 50 else "medium" if n_gt < 200
         else "dense" if n_gt < 400 else "very dense")
    d = best_dice_per_gt(VAL_BEST[image_id][0], VAL_BEST[image_id][1], INST[image_id],
                         P["t_cell"], P["t_seed"], P["min_area"])
    counts = [(d >= 0.50).sum(), ((d >= 0.35) & (d < 0.50)).sum(),
              ((d >= 0.10) & (d < 0.35)).sum(), (d < 0.10).sum()]
    for k, v in zip(band, counts):
        band[k] += int(v)
    per_bin.setdefault(b, np.zeros(4, int))
    per_bin[b] += np.array(counts, int)

tot = sum(band.values())
print("overall:", {k: f"{v / tot:.1%}" for k, v in band.items()})
for b, c in per_bin.items():
    print(f"{b:11s} {dict(zip(band, (c / c.sum()).round(3)))}")

overall: {'hit >=0.50': '77.0%', 'near 0.35-0.50': '8.6%', 'weak 0.10-0.35': '8.3%', 'missed <0.10': '6.1%'}
sparse      {'hit >=0.50': np.float64(0.844), 'near 0.35-0.50': np.float64(0.03), 'weak 0.10-0.35': np.float64(0.037), 'missed <0.10': np.float64(0.088)}
dense       {'hit >=0.50': np.float64(0.777), 'near 0.35-0.50': np.float64(0.084), 'weak 0.10-0.35': np.float64(0.086), 'missed <0.10': np.float64(0.052)}
medium      {'hit >=0.50': np.float64(0.766), 'near 0.35-0.50': np.float64(0.091), 'weak 0.10-0.35': np.float64(0.082), 'missed <0.10': np.float64(0.062)}
very dense  {'hit >=0.50': np.float64(0.735), 'near 0.35-0.50': np.float64(0.107), 'weak 0.10-0.35': np.float64(0.098), 'missed <0.10': np.float64(0.06)}


In [33]:
# ── I · FINAL STAGE 12 — RLE CHECK + SUBMISSION + REPORT ──────────────────────

from pathlib import Path
import numpy as np
import pandas as pd
import gc

# ------------------------------------------------------------------
# 0. Preflight against the objects that actually exist after G/H
# ------------------------------------------------------------------

required12 = [
    "E70_VAL",
    "E70_TEST",
    "A70_VAL",
    "A70_TEST",
    "BPOST",
    "BLEND",
    "sel",
    "val_ids",
    "TEST_IDS",
    "INST",
    "instances_from_maps",
    "flat_to_rle",
    "rle_decode_flat",
    "CFG",
    "ID_COL",
    "ANN_COL",
    "NPIX",
]

missing12 = [x for x in required12 if x not in globals()]

assert not missing12, (
    "Missing Stage 12 objects: "
    + ", ".join(missing12)
)

assert abs(float(BPOST["w"]) - 0.35) < 1e-9
assert float(sel) > 0.8112, (
    f"Stage 12 no longer clears acceptance gate: {sel:.6f}"
)

print(">>> Stage 12 final preflight passed")


# ------------------------------------------------------------------
# 1. Rebuild selected validation/test blend
# ------------------------------------------------------------------

W12 = float(BPOST["w"])

POST12 = {
    "t_cell": float(BPOST["t_cell"]),
    "t_seed": float(BPOST["t_seed"]),
    "min_area": int(BPOST["min_area"]),
    "score_thr": float(BPOST.get("score_thr", 0.0)),
}

BL12_VAL = {
    i: (
        (1.0 - W12) * E70_VAL[i]
        + W12 * A70_VAL[i]
    ).astype(np.float32)
    for i in val_ids
}

BL12_TEST = {
    i: (
        (1.0 - W12) * E70_TEST[i]
        + W12 * A70_TEST[i]
    ).astype(np.float32)
    for i in TEST_IDS
}

assert set(BL12_VAL) == set(val_ids)
assert set(BL12_TEST) == set(TEST_IDS)

print("blend weight:", W12)
print("post:", POST12)
print("selected local F1:", f"{float(sel):.6f}")


# ------------------------------------------------------------------
# 2. Exact RLE encode -> decode parity
#
# Instead of merely comparing aggregate F1, verify that every tested
# predicted mask survives the CSV codec pixel-for-pixel.
# ------------------------------------------------------------------

PARITY_N = min(25, len(val_ids))

masks_checked = 0
masks_failed = 0

for image_id in val_ids[:PARITY_N]:

    inst = instances_from_maps(
        BL12_VAL[image_id][0],
        BL12_VAL[image_id][1],
        POST12,
    )

    for flat in inst:

        encoded = flat_to_rle(flat)

        decoded = rle_decode_flat(
            encoded,
            CFG.IMG_H,
            CFG.IMG_W,
            "C",
        )

        original = np.asarray(
            flat,
            dtype=np.int64,
        )

        decoded = np.asarray(
            decoded,
            dtype=np.int64,
        )

        masks_checked += 1

        if not np.array_equal(
            original,
            decoded,
        ):
            masks_failed += 1


print(
    f"RLE parity: {masks_checked} masks checked | "
    f"{masks_failed} mismatches"
)

assert masks_failed == 0, (
    "RLE encode/decode is not lossless. "
    "Do NOT submit."
)

print(">>> RLE encoding is lossless")


# ------------------------------------------------------------------
# 3. Generate test instances
# ------------------------------------------------------------------

rows12 = []
counts12 = []
fallbacks12 = 0

for image_id in TEST_IDS:

    fg = BL12_TEST[image_id][0]
    seed = BL12_TEST[image_id][1]

    inst = instances_from_maps(
        fg,
        seed,
        POST12,
    )

    # Same safety fallback used in previous validated stages.
    if not inst:

        fallbacks12 += 1

        inst = instances_from_maps(
            fg,
            seed,
            {
                "t_cell": 0.30,
                "t_seed": 0.30,
                "min_area": 0,
                "score_thr": 0.0,
            },
        )[:1]

        if not inst:

            # Last-resort valid tiny mask.
            h = CFG.IMG_H // 2

            inst = [
                np.arange(
                    h * CFG.IMG_W,
                    h * CFG.IMG_W + 25,
                    dtype=np.int64,
                )
            ]

    counts12.append(
        len(inst)
    )

    for flat in inst:

        rows12.append(
            (
                image_id,
                flat_to_rle(flat),
            )
        )


# ------------------------------------------------------------------
# 4. Write EXACT required Stage 12 CSV
# ------------------------------------------------------------------

SUB12_PATH = Path(
    "/kaggle/working/12th_submission.csv"
)

submission12 = pd.DataFrame(
    rows12,
    columns=[
        ID_COL,
        ANN_COL,
    ],
)

submission12.insert(
    0,
    CFG.SUB_INDEX_COL,
    np.arange(
        len(submission12),
        dtype=np.int64,
    ),
)

submission12.to_csv(
    SUB12_PATH,
    index=False,
)


# ------------------------------------------------------------------
# 5. Submission integrity checks
# ------------------------------------------------------------------

assert SUB12_PATH.exists()

assert set(
    submission12[ID_COL].astype(str)
) == set(TEST_IDS), (
    "Submission is missing test images."
)

assert (
    submission12[ANN_COL]
    .notna()
    .all()
), "Submission contains empty annotations."

assert (
    submission12[
        CFG.SUB_INDEX_COL
    ].is_unique
), "Submission index is not unique."

assert len(counts12) == 182

TEST_MEAN12 = float(
    np.mean(counts12)
)

TEST_MIN12 = int(
    np.min(counts12)
)

TEST_MAX12 = int(
    np.max(counts12)
)

print("\n=== STAGE 12 SUBMISSION ===")
print("path:", SUB12_PATH)
print("rows:", len(submission12))
print(
    "predictions/image:",
    f"mean {TEST_MEAN12:.1f}",
    f"| min {TEST_MIN12}",
    f"| max {TEST_MAX12}",
)
print("fallbacks:", fallbacks12)


# ------------------------------------------------------------------
# 6. Stage 12 report
# ------------------------------------------------------------------

REPORT12_PATH = Path(
    "/kaggle/working/12th_REPORT.md"
)

weight_table12 = (
    BLEND
    .groupby("w")
    .f1
    .max()
    .round(6)
)

peak12 = float(
    BLEND.f1.max()
)

report12 = f"""# Stage 12 Report — e70 × aug70 Diverse Map Ensemble

## Strategy

Stage 12 is a zero-GPU probability-map ensemble between two ConvNeXt-Tiny
U-Net models trained on different data distributions.

### Stage 10 e70
- Standard Stage 10 training distribution
- Relative seed erosion: 0.70
- Public score: 0.5295
- Selected local F1: 0.8072
- Peak local F1: 0.8086

### Stage 11 aug70
- D4 / 90-degree rotations
- Full-image scaling before cropping
- Density-biased scale augmentation
- Dense-image oversampling
- Relative seed erosion: 0.70
- Selected local F1: 0.8012
- Peak local F1: 0.8019

Stage 11 was weaker independently, but its different training distribution
provides complementary probability maps.

## Blend Search

Weight definition:

    blended = (1-w) * e70 + w * aug70

Best F1 by weight:

{weight_table12.to_string()}

The weight axis has an interior optimum rather than being flat.

## Selected Ensemble

- aug70 weight: {W12:.2f}
- e70 weight: {1.0-W12:.2f}
- t_cell: {POST12['t_cell']}
- t_seed: {POST12['t_seed']}
- min_area: {POST12['min_area']}
- score_thr: {POST12['score_thr']}

### Validation

- Peak Stage 12 F1: {peak12:.6f}
- Selected plateau F1: {float(sel):.6f}
- Stage 10 reference: 0.8072
- Improvement over Stage 10: {float(sel)-0.8072:+.6f}

The experiment passed the pre-declared Stage 12 acceptance gate of
more than +0.004 over Stage 10.

## Near-Miss Diagnostic

Using Stage 10 e70 predictions:

- overall hit >=0.50: 77.0%
- overall near 0.35-0.50: 8.6%
- overall weak 0.10-0.35: 8.3%
- overall missed <0.10: 6.1%

Near-miss share by density:

- sparse: 3.0%
- medium: 9.1%
- dense: 8.4%
- very dense: 10.7%

This indicates that part of the remaining error, especially in medium and
very-dense images, consists of detected cells whose mask Dice lies just
below the competition's 0.50 matching threshold.

## Test Prediction Statistics

- Mean predictions/image: {TEST_MEAN12:.1f}
- Minimum: {TEST_MIN12}
- Maximum: {TEST_MAX12}
- Fallbacks: {fallbacks12}

## Submission Integrity

- RLE masks checked: {masks_checked}
- RLE mismatches: {masks_failed}
- All 182 test IDs present
- Unique submission index
- No null RLE annotations

## Output

- `/kaggle/working/12th_submission.csv`
- `/kaggle/working/12th_REPORT.md`
"""

REPORT12_PATH.write_text(
    report12,
    encoding="utf-8",
)

print("\n=== STAGE 12 REPORT ===")
print("path:", REPORT12_PATH)

print("\n>>> STAGE 12 READY")
print("submission:", SUB12_PATH)
print("report:", REPORT12_PATH)

gc.collect()

>>> Stage 12 final preflight passed
blend weight: 0.35
post: {'t_cell': 0.5, 't_seed': 0.55, 'min_area': 90, 'score_thr': 0.0}
selected local F1: 0.811945
RLE parity: 2412 masks checked | 0 mismatches
>>> RLE encoding is lossless

=== STAGE 12 SUBMISSION ===
path: /kaggle/working/12th_submission.csv
rows: 19450
predictions/image: mean 106.9 | min 14 | max 597
fallbacks: 0

=== STAGE 12 REPORT ===
path: /kaggle/working/12th_REPORT.md

>>> STAGE 12 READY
submission: /kaggle/working/12th_submission.csv
report: /kaggle/working/12th_REPORT.md


0